# 06. 분석계획 — 전체 11단계 중 6단계

작업하며 주고받은 글(내가 셀마다 적은 칸과 클로드의 답변)은 빼고 코드 · 결과 · 결정만 남긴 판이다. 진행 방식은 README 「진행 방식」에 있다.

**이 단계에서 한 일과 정한 것** — 원전 발전량은 넣어 보니 「원전 발전량이 줄면 LNG 발전량도 준다」로 배웠다. 원전이 덜 발전하면 LNG 같은 다른 발전원이 더 발전해야 하므로, 그 반대로 배운 것이다(2024년에 석탄이 줄면서 원전과 LNG가 같이 늘어서 생긴 일이다). 2024년도 더 못 맞혀서 입력에서 뺐다. 습도는 넣어도 2024년을 맞히는 정도가 거의 그대로여서 뺐다. 그다음 2023 · 2024년으로 배워 2025년에 한 번 채점하고, 2026년 1 ~ 6월과 비교하기로 했다. 모델이 이것보다 잘 맞혀야 쓸 이유가 있는 기준선(배운 해에서 같은 달 · 같은 시각 · 같은 날 종류인 시각들의 평균)과 시험할 모델 넷을 정했다. 모델이 보는 것과 보지 못하는 것은 인과지도로 그렸다.

### 말 뜻 — 표를 읽을 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 실제 | 그해(주로 2026년)에 실제로 발전한 양 · 실제 값입니다. 전력거래소 자료 그대로입니다 |
| 모델 예측 | 2023 ~ 2025년을 배운 모델이 그때의 날씨 · 시각 · 날 종류를 보고 예측한 양입니다. 9단계까지는 「평소라면」이라고 불렀습니다 |
| 실제 − 모델 예측 | 발전량 표의 숫자입니다. + 면 실제가 모델 예측보다 많았다, − 면 적었다는 뜻입니다 |
| 평소 해 | 2023 · 2024 · 2025년입니다. 「평소」는 이 세 해만 가리킵니다 |
| 평소 범위 | 평소 해에 모델 예측이 한 달 평균으로 가장 크게 틀린 크기(±2,177 MWh)입니다. 모델이 예측해 본 2024 · 2025년으로 냈습니다. 2026년의 차이가 이 안이면 「달랐다고 말할 근거가 없다」로 읽습니다 |
| 세 해 평균 | 2023 ~ 2025년 같은 달 값의 평균입니다. 모델 없이 계산한 것으로, 10단계 가격 표(2-가 · 2-라)에서만 씁니다 |

앞 단계 글에서 발전량을 두고 쓴 「평소보다 많았다 · 적었다」는 「모델 예측보다 많았다 · 적었다」로 읽습니다.

In [374]:
# 6단계 분석계획 · 준비 ①
import os                                      # 폴더와 파일 경로를 다루는 기본 모듈
import numpy as np                             # 숫자 계산
import pandas as pd                            # 표(데이터프레임)를 다루는 라이브러리
import matplotlib.pyplot as plt                # 그림을 그리는 라이브러리
from IPython.display import display            # 표를 글자가 아니라 격자로 그린다
from sklearn.linear_model import LinearRegression                                   # 5단계와 같은 선형회귀
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor   # 5단계 4-마와 같은 트리 계열 두 가지
plt.rcParams["font.family"] = "AppleGothic"    # 맥에서 그림 속 한글이 깨지지 않게
plt.rcParams["axes.unicode_minus"] = False     # 그림의 마이너스 부호가 깨지지 않게
pd.set_option("display.max_columns", 50)       # 열이 많아도 생략하지 않고 다 보이게

ROOT = os.getcwd()                             # 지금 폴더(보통 notebooks/)에서 시작해서
while not os.path.isdir(os.path.join(ROOT, "data")) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)               # data 폴더가 보일 때까지 위로 = 프로젝트 루트
HOT = os.path.join(ROOT, "data", "hot")        # 5단계가 만들어 둔 파일이 있는 폴더
RAW = os.path.join(ROOT, "data", "raw")        # 원자료 폴더 — 1-나에서 EPSIS 월별 표를 읽는다

print("프로젝트 루트:", ROOT)                    # 경로가 맞는지만 확인한다

프로젝트 루트: /Users/kimchangju/vs코드 /개인 프로젝트/lng-project


In [375]:
# 6단계 분석계획 · 준비 ②
tbl = pd.read_csv(os.path.join(HOT, "features_2023_2024_2025.csv"), encoding="utf-8-sig")  # 5단계 「저장」 셀이 만든 표
tbl["ts"] = pd.to_datetime(tbl["ts"])            # 글자로 저장된 시각을 다시 시각 자료형으로
tbl = tbl.set_index("ts").sort_index()           # 시각을 줄 이름으로 두고 시간 순서대로 놓는다
tbl.loc[tbl["year"] == 2025, "lng_mwh"] = np.nan   # 누수를 막는다 — 2025년 LNG는 7단계 마지막 채점에만 쓰므로 읽자마자 비운다(지침 v52)

학습 = tbl[tbl["year"].isin([2023, 2024])].dropna(subset=["3시간 전 기온"])   # 판단에 쓰는 두 해 — 3시간 전 기온이 빈 9시각은 뺀다
y = 학습["lng_mwh"]                               # 맞힐 값 — 그 시각의 LNG 발전량(MWh)
배움 = 학습["year"] == 2023                        # 배우는 해 — 2023년
맞힘 = 학습["year"] == 2024                        # 맞혀 보는 해 — 배우지 않은 2024년
                                                  # 2025년은 7단계 마지막 채점에만 쓴다. 이 노트북에서는 꺼내지 않는다

def 선형열(d):                                     # 선형회귀에 넣는 열 — 5단계에서 넘긴 16·24도 방식, 67열
    시각열 = pd.get_dummies(d["hour24"].astype(str), prefix="시", drop_first=True).astype(float)  # 1~23시 0/1 열 — 0시가 기준
    월열 = pd.get_dummies(d["month"].astype(str), prefix="월", drop_first=True).astype(float)     # 2~12월 0/1 열 — 1월이 기준
    날열 = pd.get_dummies(d["날 종류"], prefix="날").astype(float).drop(columns="날_월")           # 화~금·토·일·공휴일·명절 — 월요일이 기준
    쉬는날 = d["날 종류"].isin(["토", "일", "공휴일", "명절"]).astype(float)                        # 쉬는 날이면 1
    쉬는날시간대 = (시각열 - 1 / 24).mul(쉬는날, axis=0)                                           # 5단계 물음 7 — 시간대 열의 평균(1/24)을 빼고 쉬는 날과 곱한다
    쉬는날시간대.columns = [f"쉬는 날×{c[2:]}시" for c in 시각열.columns]                           # 열 이름 — 「쉬는 날×9시」
    기상 = d[["16도보다 추운 만큼", "24도보다 더운 만큼",                                           # 5단계 물음 1 — 16·24도
              "3시간 전 16도보다 추운 만큼", "3시간 전 24도보다 더운 만큼",                           # 5단계 물음 2 — 3시간 전 기온
              "icsr"]].rename(columns={"icsr": "일사"})                                          # 5단계 물음 3 — 일사 한 열
    return pd.concat([시각열, 월열, 날열, 기상, 쉬는날시간대], axis=1)

def 트리열(d):                                     # 트리 계열에 넣는 열 — 숫자 그대로 7열(5단계 물음 1·3·5)
    return d[["hour24", "month", "icsr", "ta", "dow", "is_holiday", "명절"]]

X선형 = 선형열(학습)                               # 두 해 모든 시각의 선형회귀 열
X트리 = 트리열(학습)                               # 두 해 모든 시각의 트리 계열 열

print(f"표 모양: {len(tbl):,}행 · {tbl.shape[1] + 1}열 / 판단에 쓰는 두 해 {len(학습):,}시각")
print(f"선형회귀 열 {X선형.shape[1]}개 · 트리 계열 열 {X트리.shape[1]}개 · 빈 칸 {int(X선형.isna().sum().sum() + X트리.isna().sum().sum())}개")
쓰는곳 = {2023: "배우는 해", 2024: "맞혀 보는 해", 2025: "7단계 마지막 채점 — LNG는 여기서 꺼내지 않는다"}
display(pd.DataFrame({"시각 수": tbl.groupby("year").size(),
                      "이 노트북에서": pd.Series(쓰는곳)}).rename_axis("해"))

표 모양: 25,536행 · 25열 / 판단에 쓰는 두 해 16,767시각
선형회귀 열 67개 · 트리 계열 열 7개 · 빈 칸 0개


,시각 수,이 노트북에서
해,,
2023,8760,배우는 해
2024,8016,맞혀 보는 해
2025,8760,7단계 마지막 채점 — LNG는 여기서 꺼내지 않는다


# 물음 1 — 원전을 입력에 넣을까, 넣는다면 어떻게 넣을까

> **결정** — 원전 발전량을 모델 입력에서 뺀다(2023~2026년 모두). 원전 자료는 버리지 않고 결과를 해석할 때 쓴다

## 1-가

**무엇을 보는 셀인가.** 원전만 1,000 MWh 줄였을 때 LNG 예측이 평균 얼마나 바뀌는지를, 배우는 해 넷 × 모델 셋으로 봅니다. 양수면 「원전이 줄면 LNG가 는다」로 배운 것입니다(기준 2).

In [349]:
# 6단계 분석계획 · 1-가
원전 = 학습["nuke_mwh"].rename("원전")                                        # 그 시각의 원전 발전량(MWh)
해열 = (학습["year"] == 2024).astype(float).rename("2024년이면 1")            # 두 해 사이에 달라진 것을 따로 받는 열
모델들 = {"선형회귀": (lambda: LinearRegression(), X선형),                     # 모델 이름: (모델 만들기, 넣는 열)
          "랜덤포레스트": (lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                      n_jobs=-1, random_state=0), X트리),   # 5단계 4-마와 같은 설정
          "그래디언트 부스팅": (lambda: HistGradientBoostingRegressor(early_stopping=False,
                                                                   random_state=0), X트리)}
배우는방식 = {"2023년만 배움": (배움, False), "2024년만 배움": (맞힘, False),          # 줄 이름: (배우는 시각, 「2024년이면 1」 열을 넣나)
              "두 해를 같이 배움": (배움 | 맞힘, False),
              "두 해 + 「2024년이면 1」 열": (배움 | 맞힘, True)}

def 원전만줄이면(만들기, X, 고른시각):                                         # 원전만 1,000 MWh 줄였을 때 LNG 예측이 평균 몇 MWh 바뀌나
    모델 = 만들기().fit(X[고른시각], y[고른시각])                               # 고른 해로 배운다
    줄인X = X.copy()                                                         # 넣는 열을 그대로 복사해서
    줄인X["원전"] = 줄인X["원전"] - 1000                                      # 원전만 1,000 줄이고 다른 열은 그대로 둔다
    return (모델.predict(줄인X) - 모델.predict(X)).mean()                      # 두 해 모든 시각에서 예측이 바뀐 양의 평균

표 = pd.DataFrame(index=list(배우는방식), columns=list(모델들), dtype=object)  # 줄은 배우는 방식, 칸은 모델
for 모델이름, (만들기, X) in 모델들.items():
    for 줄이름, (고른시각, 해열넣음) in 배우는방식.items():
        넣는열 = [X, 원전] + ([해열] if 해열넣음 else [])                        # 원전 한 열을 더하고, 필요하면 「2024년이면 1」 열도
        바뀜 = 원전만줄이면(만들기, pd.concat(넣는열, axis=1), 고른시각)
        표.loc[줄이름, 모델이름] = f"{바뀜:+,.0f}"                              # 글자로 — 부호와 천 단위 쉼표

print("[표 1] 원전만 1,000 MWh 줄이고 다른 열은 그대로 두었을 때 LNG 예측이 바뀐 양 (MWh)")
print("두 해 모든 시각의 평균 · 양수면 원전이 줄 때 LNG 예측이 늘었다 · 선형회귀는 「원전 기울기 × −1,000」과 같다")
display(표)

[표 1] 원전만 1,000 MWh 줄이고 다른 열은 그대로 두었을 때 LNG 예측이 바뀐 양 (MWh)
두 해 모든 시각의 평균 · 양수면 원전이 줄 때 LNG 예측이 늘었다 · 선형회귀는 「원전 기울기 × −1,000」과 같다


,선형회귀,랜덤포레스트,그래디언트 부스팅
2023년만 배움,+315,-204,-268
2024년만 배움,+124,-6,+268
두 해를 같이 배움,-206,-156,-221
두 해 + 「2024년이면 1」 열,+97,-51,+10


## 1-나

**무엇을 보는 셀인가.** EPSIS 월별 표로 2023→2024년에 원자력·석탄·LNG·나머지가 한 시각 평균 얼마나 달라졌는지 봅니다. 2024년 8월은 자료가 없어 두 해 모두 뺍니다.

In [350]:
# 6단계 분석계획 · 1-나
ep = pd.read_csv(os.path.join(RAW, "epsis_fuel_monthly.csv"), encoding="cp949",
                 header=None, skiprows=2)                                  # EPSIS 월별 연료원별 전력거래량(GWh) — 머리글 두 줄은 건너뛴다
ep = ep[[0, 1, 4, 6, 18]]                                                 # 열 위치로 고른다 — 0 기간 · 1 원자력 · 4 석탄 계 · 6 LNG · 18 전력시장 계
ep.columns = ["기간", "원자력", "석탄", "LNG", "전력시장 전체"]
ep["해"] = ep["기간"].str[:4].astype(int)                                 # 「2024/03」에서 앞 네 자리
ep["월"] = ep["기간"].str[5:7].astype(int)                                # 뒤 두 자리
ep = ep[ep["해"].isin([2023, 2024]) & (ep["월"] != 8)]                     # 두 해만, 8월은 두 해에서 똑같이 뺀다(2024년 8월이 우리 자료에 없다)
ep["시각 수"] = pd.to_datetime(ep["기간"] + "/01", format="%Y/%m/%d").dt.days_in_month * 24   # 그 달의 시각 수 — 2024년 2월은 29일이라 24시각 많다
합 = ep.groupby("해")[["원자력", "석탄", "LNG", "전력시장 전체", "시각 수"]].sum()   # 해마다 11개월을 더한다
평균 = 합.drop(columns="시각 수").div(합["시각 수"], axis=0) * 1000            # GWh를 시각 수로 나눠 한 시각 평균 MWh로
평균["나머지 (재생에너지·양수·유류 등)"] = 평균["전력시장 전체"] - 평균[["원자력", "석탄", "LNG"]].sum(axis=1)
평균 = 평균[["원자력", "석탄", "LNG", "나머지 (재생에너지·양수·유류 등)", "전력시장 전체"]].T   # 줄은 연료, 칸은 해
평균["차이 (2024 − 2023)"] = 평균[2024] - 평균[2023]
평균["차이 (%)"] = 평균["차이 (2024 − 2023)"] / 평균[2023] * 100
표 = pd.DataFrame({"2023년": 평균[2023].map("{:,.0f}".format),                # 글자로 — 천 단위 쉼표
                   "2024년": 평균[2024].map("{:,.0f}".format),
                   "차이 (2024 − 2023)": 평균["차이 (2024 − 2023)"].map("{:+,.0f}".format),
                   "차이 (%)": 평균["차이 (%)"].map("{:+.1f}".format)})

print("[표 1] 8월을 뺀 11개월, 연료마다 한 시각 평균 전력거래량 (MWh) — EPSIS 월별 표")
print("2024년 2월은 29일까지 들어 있어서 합계가 아니라 한 시각 평균으로 비교한다")
display(표)

[표 1] 8월을 뺀 11개월, 연료마다 한 시각 평균 전력거래량 (MWh) — EPSIS 월별 표
2024년 2월은 29일까지 들어 있어서 합계가 아니라 한 시각 평균으로 비교한다


,2023년,2024년,차이 (2024 − 2023),차이 (%)
원자력,"19,519","20,399",+880,+4.5
석탄,"20,069","17,780","-2,289",-11.4
LNG,"17,365","18,289",+924,+5.3
나머지 (재생에너지·양수·유류 등),"4,603","5,161",+558,+12.1
전력시장 전체,"61,557","61,629",+73,+0.1


## 1-다

**무엇을 보는 셀인가.** 2023년으로 배워 2024년을 맞힐 때, 원전을 넣기 전과 뒤의 R²(기준 1)와 오차를 세 모델로 봅니다. [표 2]의 평균 수준 오차는 「실제 − 예측」의 평균이고, 양수면 2024년을 낮게 예측했다는 뜻입니다.

In [351]:
# 6단계 분석계획 · 1-다
원전 = 학습["nuke_mwh"].rename("원전")                                        # 1-가와 같은 원전 열
R2표 = pd.DataFrame(index=list(모델들), columns=["넣기 전", "넣은 뒤"], dtype=float)   # 줄은 모델, 칸은 원전을 넣기 전·후
오차표 = {}                                                                 # 모델마다 오차를 둘로 나눈 값
for 모델이름, (만들기, X) in 모델들.items():                                   # 1-가의 세 모델
    for 칸, XX in [("넣기 전", X), ("넣은 뒤", pd.concat([X, 원전], axis=1))]:
        모델 = 만들기().fit(XX[배움], y[배움])                                   # 2023년으로만 배운다
        틀림 = y[맞힘] - 모델.predict(XX[맞힘])                                  # 2024년 시각마다 실제 − 예측
        R2표.loc[모델이름, 칸] = 1 - (틀림 ** 2).sum() / ((y[맞힘] - y[맞힘].mean()) ** 2).sum()   # 2024년을 맞힌 R²
        오차표[(모델이름, f"평균 수준 오차 · {칸}")] = f"{틀림.mean():+,.0f}"                  # 2024년 모든 시각의 틀린 양 평균
        오차표[(모델이름, f"평균을 뺀 뒤 남는 오차 · {칸}")] = f"{틀림.std(ddof=0):,.0f}"       # 그 평균을 빼고 남은 오르내림의 크기
R2표["차이 (넣은 뒤 − 넣기 전)"] = R2표["넣은 뒤"].round(4) - R2표["넣기 전"].round(4)
R2표 = R2표.apply(lambda 칸: 칸.map("{:.4f}".format))                         # 글자로 — 소수 넷째 자리
오차표 = pd.Series(오차표).unstack()[[f"평균 수준 오차 · 넣기 전", f"평균 수준 오차 · 넣은 뒤",
                                      f"평균을 뺀 뒤 남는 오차 · 넣기 전", f"평균을 뺀 뒤 남는 오차 · 넣은 뒤"]].loc[list(모델들)]

print("[표 1] 2023년으로만 배워 2024년을 맞힌 R² — 원전 한 열을 넣기 전과 넣은 뒤")
display(R2표)
print("[표 2] 같은 예측의 오차를 둘로 나눈 것 (MWh)")
print("평균 수준 오차 = 2024년 모든 시각의 「실제 − 예측」 평균(양수면 낮게 예측) · 남는 오차 = 그 평균을 빼고 남은 것의 표준편차")
display(오차표)

[표 1] 2023년으로만 배워 2024년을 맞힌 R² — 원전 한 열을 넣기 전과 넣은 뒤


,넣기 전,넣은 뒤,차이 (넣은 뒤 − 넣기 전)
선형회귀,0.7524,0.7267,-0.0257
랜덤포레스트,0.7618,0.7133,-0.0485
그래디언트 부스팅,0.7807,0.7480,-0.0327


[표 2] 같은 예측의 오차를 둘로 나눈 것 (MWh)
평균 수준 오차 = 2024년 모든 시각의 「실제 − 예측」 평균(양수면 낮게 예측) · 남는 오차 = 그 평균을 빼고 남은 것의 표준편차


,평균 수준 오차 · 넣기 전,평균 수준 오차 · 넣은 뒤,평균을 뺀 뒤 남는 오차 · 넣기 전,평균을 뺀 뒤 남는 오차 · 넣은 뒤
선형회귀,+895,"+1,192","2,564","2,593"
랜덤포레스트,+821,+887,"2,534","2,785"
그래디언트 부스팅,+884,+865,"2,398","2,600"


## 1-라

**무엇을 보는 셀인가.** 「결정까지의 순서」 3번(왜 그렇게 나왔나)을 봅니다. [표 1]은 한 해 안에서 달마다 원전과 LNG가 어떻게 움직였는지, [표 2]는 원전 열 때문에 2024년 예측이 얼마나 바뀌었는지입니다.

In [352]:
# 6단계 분석계획 · 1-라
원전 = 학습["nuke_mwh"].rename("원전")                                          # 1-가·1-다와 같은 원전 열
달평균 = 학습.pivot_table(index="month", columns="year",
                          values=["nuke_mwh", "lng_mwh"], aggfunc="mean")       # 해·달마다 한 시각 평균(MWh) — 2024년 8월은 자료가 없다
표1 = pd.DataFrame({f"{이름} {해}": 달평균[(열, 해)]                            # 칸 넷 — 해마다 원전과 LNG
                    for 해 in (2023, 2024) for 이름, 열 in [("원전", "nuke_mwh"), ("LNG", "lng_mwh")]})
표1.index.name = "달"                                                           # 줄 이름 — 1~12월
상관 = {해: 표1[f"원전 {해}"].corr(표1[f"LNG {해}"]) for 해 in (2023, 2024)}     # 달 평균끼리의 상관 — 해마다 따로

X원전 = pd.concat([X선형, 원전], axis=1)                                         # 1-다 「넣은 뒤」와 같은 열
넣기전 = LinearRegression().fit(X선형[배움], y[배움])                             # 원전 없이 2023년으로만 배운 선형회귀
넣은뒤 = LinearRegression().fit(X원전[배움], y[배움])                             # 원전을 넣고 2023년으로만 배운 선형회귀
기울기 = 넣은뒤.coef_[X원전.columns.get_loc("원전")]                              # 원전 1 MWh당 LNG 예측이 바뀌는 양
원전차 = 원전[맞힘].mean() - 원전[배움].mean()                                    # 2024년 원전 평균 − 2023년 원전 평균
틀림전 = (y[맞힘] - 넣기전.predict(X선형[맞힘])).mean()                           # 1-다 「평균 수준 오차 · 넣기 전」과 같은 값
틀림후 = (y[맞힘] - 넣은뒤.predict(X원전[맞힘])).mean()                           # 1-다 「평균 수준 오차 · 넣은 뒤」와 같은 값
표2 = pd.Series({"① 2023년으로 배운 원전 기울기 (원전 1 MWh당 LNG)": f"{기울기:+.3f}",
                 "② 원전 평균 — 2024년 − 2023년 (MWh)": f"{원전차:+,.0f}",
                 "③ ① × ② — 원전 열 때문에 2024년 LNG 예측이 바뀐 양 (MWh)": f"{기울기 * 원전차:+,.0f}",
                 "④ 1-다 평균 수준 오차 — 넣기 전 → 넣은 뒤 (MWh)": f"{틀림전:+,.0f} → {틀림후:+,.0f} (차이 {틀림후 - 틀림전:+,.0f})"},
                name="값").to_frame()                                           # 글자로 — 부호와 천 단위 쉼표

print("[표 1] 달마다 원전과 LNG의 한 시각 평균 (MWh) — 2024년 8월은 자료가 없어 비어 있다")
display(표1.apply(lambda 칸: 칸.map(lambda v: "" if pd.isna(v) else f"{v:,.0f}")))
print(f"달 평균끼리의 상관 — 2023년 {상관[2023]:+.2f} · 2024년 {상관[2024]:+.2f} (2024년은 8월을 뺀 11개 달)")
print("[표 2] 원전을 넣으면 2024년을 왜 덜 맞히나 — 선형회귀")
display(표2)

[표 1] 달마다 원전과 LNG의 한 시각 평균 (MWh) — 2024년 8월은 자료가 없어 비어 있다


,원전 2023,LNG 2023,원전 2024,LNG 2024
달,,,,
1,"20,047","20,225","19,108","21,070"
2,"19,289","19,809","20,113","18,261"
3,"19,446","18,519","19,936","19,350"
4,"17,925","16,043","19,321","16,186"
5,"18,484","15,365","20,488","16,087"
6,"18,705","16,231","19,891","17,682"
7,"20,217","18,237","21,056","20,168"
8,"20,274","19,229",,
9,"19,782","17,385","20,911","18,798"


달 평균끼리의 상관 — 2023년 +0.47 · 2024년 -0.16 (2024년은 8월을 뺀 11개 달)
[표 2] 원전을 넣으면 2024년을 왜 덜 맞히나 — 선형회귀


,값
① 2023년으로 배운 원전 기울기 (원전 1 MWh당 LNG),-0.315
② 원전 평균 — 2024년 − 2023년 (MWh),+885
③ ① × ② — 원전 열 때문에 2024년 LNG 예측이 바뀐 양 (MWh),-279
④ 1-다 평균 수준 오차 — 넣기 전 → 넣은 뒤 (MWh),"+895 → +1,192 (차이 +296)"


# 물음 2 — 모델이 단순한 방법(기준선)보다 나은가

> **결정** — 낫다. 2023년으로 배워 2024년을 맞힌 R²가 모델 셋 모두 가장 나은 기준선보다 0.0587~0.0866 높아서, 이 방향으로 7단계까지 간다

## 2-나

**무엇을 보는 셀인가.** 2023년으로 배워 2024년을 맞힌 R²를 기준선 넷과 모델 셋에서 나란히 봅니다([표 1]). 같은 예측의 평균 절대 오차는 9단계에 쓰려고 따로 적습니다([표 2]).

In [353]:
# 6단계 분석계획 · 2-나
쉬는날 = tbl["날 종류"].isin(["토", "일", "공휴일", "명절"])                      # 5단계와 같은 쉬는 날 — 토·일·공휴일·명절
y23 = tbl.loc[tbl["year"] == 2023, "lng_mwh"]                                     # 기준선이 쓰는 2023년 LNG — 8,760시각 모두
t24 = 학습.index[맞힘]                                                            # 맞혀 보는 2024년 시각
전 = t24 - pd.Timedelta(days=364)                                                 # 52주 전 — 요일이 같은 날
있음 = 전.isin(y23.index)                                                         # 52주 전이 2023년에 있는 시각만 쓴다 — 12월 30·31일은 없다
볼시각 = t24[있음]                                                                # 일곱 방법을 모두 같은 시각에서 비교한다
실제 = y.loc[볼시각]                                                              # 그 시각의 실제 LNG(MWh)

예측 = {}                                                                         # 방법 이름: 그 시각들의 예측
예측["① 한 해 평균"] = pd.Series(y23.mean(), index=볼시각)                         # 2023년 모든 시각의 평균 하나
시각평균 = y23.groupby(tbl.loc[y23.index, "hour24"]).mean()                        # 2023년 시각(0~23시)마다 평균
예측["② 시각별 평균"] = pd.Series(시각평균.loc[tbl.loc[볼시각, "hour24"]].values, index=볼시각)
키23 = [tbl.loc[y23.index, "month"], tbl.loc[y23.index, "hour24"], 쉬는날.loc[y23.index]]   # 2023년 시각마다 (달, 시각, 쉬는 날인가)
셋평균 = y23.groupby(키23).mean()                                                  # 2023년 같은 달·같은 시각·같은 날 종류의 평균
키24 = pd.MultiIndex.from_arrays([tbl.loc[볼시각, "month"], tbl.loc[볼시각, "hour24"], 쉬는날.loc[볼시각]])
예측["③ 달·시각·날별 평균"] = pd.Series(셋평균.reindex(키24).values, index=볼시각)
예측["④ 52주 전 같은 요일"] = pd.Series(y23.loc[전[있음]].values, index=볼시각)     # 364일 전 같은 시각의 2023년 값

모델들 = {"선형회귀": (lambda: LinearRegression(), X선형),                         # 물음 1과 같은 세 모델 — 원전 열은 없다
          "랜덤포레스트": (lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                      n_jobs=-1, random_state=0), X트리),
          "그래디언트 부스팅": (lambda: HistGradientBoostingRegressor(early_stopping=False,
                                                                   random_state=0), X트리)}
for 이름, (만들기, X) in 모델들.items():
    모델 = 만들기().fit(X[배움], y[배움])                                          # 2023년으로만 배운다
    예측[이름] = pd.Series(모델.predict(X.loc[볼시각]), index=볼시각)                # 2024년을 맞힌다

def R2(p):                                                                        # 2024년을 맞힌 R² — 1-다와 같은 식
    return 1 - ((실제 - p) ** 2).sum() / ((실제 - 실제.mean()) ** 2).sum()
R2값 = pd.Series({이름: R2(p) for 이름, p in 예측.items()})                        # 일곱 방법의 R²
기준최고 = R2값.iloc[:4].max()                                                     # 기준선 넷 가운데 가장 높은 R²
표1 = pd.DataFrame({"2024년 R²": R2값.map("{:.4f}".format),
                    "기준선 중 가장 높은 R²와의 차이": (R2값.round(4) - round(기준최고, 4)).map("{:+.4f}".format)})
오차 = pd.Series({이름: (실제 - p).abs().mean() for 이름, p in 예측.items()})       # 한 시각에 평균 몇 MWh 틀렸나
표2 = 오차.map("{:,.0f}".format).rename("평균 절대 오차 (MWh)").to_frame()

print(f"2024년 {len(볼시각):,}시각에서 비교 — 12월 30·31일 {int((~있음).sum())}시각은 ④의 52주 전이 2023년에 없어 일곱 방법 모두에서 뺐다")
print("[표 1] 2023년으로 배워 2024년을 맞힌 R² — 기준선 넷과 모델 셋")
display(표1)
print("[표 2] 같은 예측의 평균 절대 오차 — 한 시각에 평균 몇 MWh 틀렸나(작을수록 덜 틀렸다 · 9단계용으로 적어 둔다)")
display(표2)

2024년 7,962시각에서 비교 — 12월 30·31일 48시각은 ④의 52주 전이 2023년에 없어 일곱 방법 모두에서 뺐다
[표 1] 2023년으로 배워 2024년을 맞힌 R² — 기준선 넷과 모델 셋


,2024년 R²,기준선 중 가장 높은 R²와의 차이
① 한 해 평균,-0.0187,-0.7135
② 시각별 평균,0.3060,-0.3888
③ 달·시각·날별 평균,0.6948,+0.0000
④ 52주 전 같은 요일,0.5249,-0.1699
선형회귀,0.7535,+0.0587
랜덤포레스트,0.7628,+0.0680
그래디언트 부스팅,0.7814,+0.0866


[표 2] 같은 예측의 평균 절대 오차 — 한 시각에 평균 몇 MWh 틀렸나(작을수록 덜 틀렸다 · 9단계용으로 적어 둔다)


,평균 절대 오차 (MWh)
① 한 해 평균,"4,449"
② 시각별 평균,"3,522"
③ 달·시각·날별 평균,"2,341"
④ 52주 전 같은 요일,"2,832"
선형회귀,"2,142"
랜덤포레스트,"2,073"
그래디언트 부스팅,"2,000"


## 2-다

**무엇을 보는 셀인가.** 2-나와 같은 예측의 평균 절대 오차를 달마다([표 1]), 하루를 네 시간씩 나눈 묶음마다([표 2]) 봅니다. 달마다 R²는 그 달 안의 오르내림으로 계산되어 달끼리 비교하기 어려워서, 나눠 볼 때는 평균 절대 오차를 씁니다. 기준선은 넷 가운데 2024년 R²가 가장 높은 것 하나만 놓습니다.

In [354]:
# 6단계 분석계획 · 2-다
최고기준 = R2값.iloc[:4].idxmax()                                                  # 기준선 넷 가운데 2024년 R²가 가장 높은 것 — 2-나 [표 1]
고른 = [최고기준, "선형회귀", "랜덤포레스트", "그래디언트 부스팅"]                    # 이 셀에서 나란히 볼 넷
틀림 = pd.DataFrame({이름: (실제 - 예측[이름]).abs() for 이름 in 고른})              # 시각마다 틀린 크기(부호를 뗀다)
달 = tbl.loc[볼시각, "month"].rename("달")                                         # 그 시각의 달 — 2024년 8월은 자료가 없다
묶음 = pd.cut(tbl.loc[볼시각, "hour24"], bins=[-1, 3, 7, 11, 15, 19, 23],
              labels=["0~3시", "4~7시", "8~11시", "12~15시", "16~19시", "20~23시"]).rename("시각")  # 하루를 네 시간씩 여섯 묶음
표1 = 틀림.groupby(달).mean()                                                      # 달마다 평균 절대 오차
표2 = 틀림.groupby(묶음, observed=True).mean()                                     # 시각 묶음마다 평균 절대 오차
쉼표 = lambda 표: 표.apply(lambda 칸: 칸.map("{:,.0f}".format))                     # 천 단위 쉼표

print(f"기준선은 2-나 [표 1]에서 R²가 가장 높은 「{최고기준}」 하나만 놓았다 · 단위 MWh · 작을수록 덜 틀렸다")
print("[표 1] 달마다 평균 절대 오차 — 2024년 8월은 자료가 없다")
display(쉼표(표1))
print("[표 2] 하루를 네 시간씩 나눈 묶음마다 평균 절대 오차")
display(쉼표(표2))

기준선은 2-나 [표 1]에서 R²가 가장 높은 「③ 달·시각·날별 평균」 하나만 놓았다 · 단위 MWh · 작을수록 덜 틀렸다
[표 1] 달마다 평균 절대 오차 — 2024년 8월은 자료가 없다


,③ 달·시각·날별 평균,선형회귀,랜덤포레스트,그래디언트 부스팅
달,,,,
1,"2,200","2,113","1,670","1,581"
2,"2,701","2,146","2,308","2,215"
3,"2,401","1,967","1,900","1,770"
4,"1,845","1,984","1,782","1,752"
5,"1,911","1,958","1,807","1,811"
6,"2,210","2,073","1,999","1,947"
7,"2,234","2,376","2,172","2,092"
9,"3,158","2,171","2,633","2,282"
10,"2,493","2,165","2,234","2,216"


[표 2] 하루를 네 시간씩 나눈 묶음마다 평균 절대 오차


,③ 달·시각·날별 평균,선형회귀,랜덤포레스트,그래디언트 부스팅
시각,,,,
0~3시,"2,165","2,111","2,018","1,967"
4~7시,"2,101","1,973","1,968","1,903"
8~11시,"2,510","2,158","2,038","1,985"
12~15시,"2,697","2,127","2,150","1,974"
16~19시,"2,334","2,270","2,157","2,099"
20~23시,"2,238","2,212","2,105","2,071"


# 물음 3 — 2025·2026년 기온·일사가 2023·2024년에 본 범위 안에 있나, 일사 한 열로 설비가 는 것을 따라가나

> **결정** — 2025·2026년 날씨는 거의 다 배운 범위 안이고(2025년 30시각, 2026년 1~6월 1시각만 밖), 2024년에는 일사 한 열이 늘어난 패널을 못 따라간다는 흔적이 없다. 그래서 바꿀 것 없이 이대로 7단계로 간다

## 3-나

**무엇을 보는 셀인가.** 기온·일사마다 2023·2024년(배운 범위)과 2025년·2026년 1~6월의 가장 낮은 값 ~ 가장 높은 값([표 1]), 그리고 배운 범위 밖인 시각 수([표 2])입니다. 2025·2026년은 기온·일사만 읽고 LNG는 읽지 않습니다.

In [355]:
# 6단계 분석계획 · 3-나
import glob                                                                        # 이름 규칙으로 여러 파일을 한 번에 찾는다
본 = 학습[["ta", "icsr"]]                                                           # 모델이 배울 때 본 값 — 2023년과 2024년(8월·2월 29일 빼고)
기상25 = tbl.loc[tbl["year"] == 2025, ["ta", "icsr"]]                               # 2025년 기온·일사 — 2025년 LNG는 준비 ②에서 비웠다
파일들 = sorted(glob.glob(os.path.join(RAW, "asos", "asos_108_2026*.csv")))          # 2026년 기상청 서울 108 원자료 — LNG 파일은 열지 않는다
기상26 = pd.concat([pd.read_csv(f, encoding="utf-8-sig", usecols=["tm", "ta", "icsr"]) for f in 파일들])   # 시각·기온·일사 세 칸만 읽는다
기상26["tm"] = pd.to_datetime(기상26["tm"])                                        # 글자로 된 시각을 시각 자료형으로
기상26 = 기상26[(기상26["tm"] >= "2026-01-01") & (기상26["tm"] < "2026-07-01")].set_index("tm")   # 9단계에서 비교할 1~6월만
기상26["icsr"] = 기상26["icsr"].fillna(0)                                          # 밤에 비어 있는 일사는 0 — 5단계와 같다
해들 = {"2025년": 기상25, "2026년 1~6월": 기상26[["ta", "icsr"]]}                    # 범위를 볼 두 해

def 범위(d):                                                                       # 가장 낮은 값 ~ 가장 높은 값
    return [f"{d['ta'].min():.1f} ~ {d['ta'].max():.1f}", f"{d['icsr'].min():.2f} ~ {d['icsr'].max():.2f}"]
표1 = pd.DataFrame({"2023·2024년(배운 범위)": 범위(본)} | {이름: 범위(d) for 이름, d in 해들.items()},
                   index=["기온 (℃)", "일사 (MJ/m²)"])                              # 줄은 기온·일사, 칸은 해

줄 = {}                                                                            # 해마다 범위 밖 시각 수를 모은다
for 이름, d in 해들.items():
    낮음 = d["ta"] < 본["ta"].min()                                                 # 배운 가장 추운 시각보다 더 추움
    높음 = d["ta"] > 본["ta"].max()                                                 # 배운 가장 더운 시각보다 더 더움
    일사 = d["icsr"] > 본["icsr"].max()                                             # 배운 가장 센 일사보다 더 셈
    밖 = 낮음 | 높음 | 일사                                                          # 셋 가운데 하나라도 해당하는 시각
    줄[이름] = {"시각 수": f"{len(d):,}", "기온이 더 낮음": int(낮음.sum()), "기온이 더 높음": int(높음.sum()),
               "일사가 더 셈": int(일사.sum()), "범위 밖 시각": int(밖.sum()), "범위 밖 비율": f"{밖.mean():.2%}"}
표2 = pd.DataFrame(줄).T.rename_axis("해")                                          # 줄은 해, 칸은 무엇이 벗어났나

print("[표 1] 기온·일사의 범위(가장 낮은 값 ~ 가장 높은 값) — 2025·2026년은 기온·일사만 읽었고 LNG는 읽지 않았다")
display(표1)
print("[표 2] 2023·2024년에 본 범위 밖인 시각 수 — 「범위 밖 시각」은 셋 가운데 하나라도 해당하는 시각")
display(표2)

[표 1] 기온·일사의 범위(가장 낮은 값 ~ 가장 높은 값) — 2025·2026년은 기온·일사만 읽었고 LNG는 읽지 않았다


,2023·2024년(배운 범위),2025년,2026년 1~6월
기온 (℃),-17.2 ~ 35.8,-12.1 ~ 37.6,-13.1 ~ 33.3
일사 (MJ/m²),0.00 ~ 3.81,0.00 ~ 3.82,0.00 ~ 3.90


[표 2] 2023·2024년에 본 범위 밖인 시각 수 — 「범위 밖 시각」은 셋 가운데 하나라도 해당하는 시각


,시각 수,기온이 더 낮음,기온이 더 높음,일사가 더 셈,범위 밖 시각,범위 밖 비율
해,,,,,,
2025년,"8,760",0,29,1,30,0.34%
2026년 1~6월,"4,344",0,0,1,1,0.02%


## 3-다

**무엇을 보는 셀인가.** 해마다 태양광 설비용량 평균([표 1])과, 2-나에서 2023년으로 배워 2024년을 맞힌 세 모델의 잔차를 하루 다섯 시간 묶음으로 나눈 방향([표 2], 평균 잔차)과 크기([표 3], 평균 절대 오차)입니다. 2-나를 먼저 돌려야 합니다.

In [356]:
# 6단계 분석계획 · 3-다
설비 = tbl[tbl["month"] != 8].groupby("year")["solar_cap_mw"].mean()                # 해마다 태양광 설비용량 평균(MW) — 2024년 8월이 없어서 세 해 모두 8월을 빼고 낸다
표1 = pd.DataFrame({"태양광 설비용량 평균 (MW)": 설비.map("{:,.0f}".format),
                    "2023년보다 는 것 (MW)": (설비 - 설비[2023]).map("{:+,.0f}".format)}).rename_axis("해")   # 입력 쪽 값이라 2025년도 봐도 된다

모델셋 = ["선형회귀", "랜덤포레스트", "그래디언트 부스팅"]                            # 2-나의 세 모델 — 2-나를 먼저 돌려야 한다
잔차 = pd.DataFrame({이름: 실제 - 예측[이름] for 이름 in 모델셋})                      # 2024년 시각마다 LNG 잔차(실제 LNG − 예측 LNG)
시 = tbl.loc[볼시각, "hour24"]                                                     # 그 시각이 몇 시인가
묶음 = pd.cut(시, bins=[-1, 5, 9, 15, 19, 23],
              labels=["0~5시 (밤)", "6~9시", "10~15시 (낮)", "16~19시", "20~23시"]).rename("시각")   # 하루를 다섯 묶음으로
표2 = 잔차.groupby(묶음, observed=True).mean()                                      # 방향 — 묶음마다 평균 잔차
표3 = 잔차.abs().groupby(묶음, observed=True).mean()                                # 크기 — 묶음마다 평균 절대 오차

print("[표 1] 해마다 태양광 설비용량 평균 — 2024년 8월 자료가 없어서 세 해 모두 8월을 빼고 냈다")
display(표1)
print("[표 2] 방향 — LNG 예측의 평균 잔차(실제 LNG − 예측 LNG, MWh). 2023년으로 배워 2024년을 맞혔다. 양수면 실제보다 낮게, 음수면 높게 예측했다")
display(표2.apply(lambda 칸: 칸.map("{:+,.0f}".format)))
print("[표 3] 크기 — 같은 LNG 예측의 평균 절대 오차(MWh). 작을수록 덜 틀렸다")
display(표3.apply(lambda 칸: 칸.map("{:,.0f}".format)))

[표 1] 해마다 태양광 설비용량 평균 — 2024년 8월 자료가 없어서 세 해 모두 8월을 빼고 냈다


,태양광 설비용량 평균 (MW),2023년보다 는 것 (MW)
해,,
2023,"7,814",+0
2024,"8,998","+1,185"
2025,"10,052","+2,238"


[표 2] 방향 — LNG 예측의 평균 잔차(실제 LNG − 예측 LNG, MWh). 2023년으로 배워 2024년을 맞혔다. 양수면 실제보다 낮게, 음수면 높게 예측했다


,선형회귀,랜덤포레스트,그래디언트 부스팅
시각,,,
0~5시 (밤),+861,+679,+667
6~9시,+954,+748,+830
10~15시 (낮),+755,+803,+922
16~19시,+958,"+1,081","+1,160"
20~23시,+967,+814,+877


[표 3] 크기 — 같은 LNG 예측의 평균 절대 오차(MWh). 작을수록 덜 틀렸다


,선형회귀,랜덤포레스트,그래디언트 부스팅
시각,,,
0~5시 (밤),"2,091","2,034","1,968"
6~9시,"2,023","1,929","1,892"
10~15시 (낮),"2,140","2,129","1,990"
16~19시,"2,270","2,157","2,099"
20~23시,"2,212","2,105","2,071"


## 3-라

**무엇을 보는 셀인가.** 2024년 날마다 「낮(10~15시) 평균 잔차 − 밤(0~5시) 평균 잔차」를 구하고, 모델마다 그 평균과 95% 신뢰구간을 봅니다. 3-다를 먼저 돌려야 합니다.

In [357]:
# 6단계 분석계획 · 3-라
날 = pd.Series(볼시각.normalize(), index=볼시각)                                     # 그 시각이 어느 날인가
낮시각 = 시.between(10, 15)                                                          # 낮 10~15시
밤시각 = 시.between(0, 5)                                                            # 밤 0~5시
낮날 = 잔차[낮시각].groupby(날[낮시각]).mean()                                       # 날마다 낮 평균 잔차 — 모델 셋
밤날 = 잔차[밤시각].groupby(날[밤시각]).mean()                                       # 날마다 밤 평균 잔차
차이 = (낮날 - 밤날).dropna()                                                         # 날마다 「낮 평균 잔차 − 밤 평균 잔차」 — 낮과 밤이 다 있는 날만
날수 = len(차이)                                                                     # 비교에 들어간 날 수
평균 = 차이.mean()                                                                   # 모델마다 그 차이의 평균
표준오차 = 차이.std(ddof=1) / np.sqrt(날수)                                          # 표준편차 ÷ √날 수
표1 = pd.DataFrame({"날마다 낮 − 밤의 평균": 평균.map("{:+,.0f}".format),
                    "표준오차": 표준오차.map("{:,.0f}".format),
                    "95% 구간 아래 끝": (평균 - 1.96 * 표준오차).map("{:+,.0f}".format),
                    "95% 구간 위 끝": (평균 + 1.96 * 표준오차).map("{:+,.0f}".format)}).rename_axis("모델")

print(f"2024년 {날수}일 — 날마다 LNG 예측의 낮(10~15시) 평균 잔차에서 밤(0~5시) 평균 잔차를 뺐다 · 잔차 = 실제 LNG − 예측 LNG · 단위 MWh")
print("[표 1] 그 차이의 평균과 95% 신뢰구간(평균 ± 1.96 × 표준오차) — 위 끝까지 0보다 아래면 「낮을 한쪽으로 틀린다」(3-가에서 정한 기준)")
display(표1)

2024년 332일 — 날마다 LNG 예측의 낮(10~15시) 평균 잔차에서 밤(0~5시) 평균 잔차를 뺐다 · 잔차 = 실제 LNG − 예측 LNG · 단위 MWh
[표 1] 그 차이의 평균과 95% 신뢰구간(평균 ± 1.96 × 표준오차) — 위 끝까지 0보다 아래면 「낮을 한쪽으로 틀린다」(3-가에서 정한 기준)


,날마다 낮 − 밤의 평균,표준오차,95% 구간 아래 끝,95% 구간 위 끝
모델,,,,
선형회귀,-93,174,-434,+248
랜덤포레스트,+132,170,-201,+465
그래디언트 부스팅,+263,151,-33,+559


## 3-마

**무엇을 보는 셀인가.** 3-라의 날마다 낮 − 밤 차이를 달마다 평균 낸 것([표 1])과, 11달 가운데 0보다 아래인 달 수([표 2])입니다. 3-라를 먼저 돌려야 합니다.

In [358]:
# 6단계 분석계획 · 3-마
달 = pd.Series(차이.index.month, index=차이.index).rename("달")                      # 그 날이 몇 월인가 — 2024년 8월은 자료가 없다
달평균 = 차이.groupby(달).mean()                                                     # 달마다 「낮 − 밤」의 평균 — 모델 셋
날수달 = 차이.groupby(달).size()                                                      # 달마다 들어간 날 수
표1 = 달평균.apply(lambda 칸: 칸.map("{:+,.0f}".format))                              # 부호와 천 단위 쉼표를 붙인다
표1.index = [f"{m}월" for m in 표1.index]                                            # 줄 이름을 「3월」처럼
표1 = 표1.rename_axis("달")
아래 = (달평균 < 0).sum()                                                            # 모델마다 0보다 아래인 달 수
표2 = pd.DataFrame({"0보다 아래인 달": 아래.map(lambda n: f"{n}달 / {len(달평균)}달")}).rename_axis("모델")

print(f"2024년 {len(달평균)}달 — 8월은 자료가 없다 · 달마다 {날수달.min()}~{날수달.max()}일 · 단위 MWh")
print("[표 1] 달마다 LNG 예측의 「낮(10~15시) 평균 잔차 − 밤(0~5시) 평균 잔차」 평균 — 3-라의 날마다 차이를 달마다 모았다")
display(표1)
print(f"[표 2] {len(달평균)}달 가운데 0보다 아래인 달 수")
display(표2)

2024년 11달 — 8월은 자료가 없다 · 달마다 28~31일 · 단위 MWh
[표 1] 달마다 LNG 예측의 「낮(10~15시) 평균 잔차 − 밤(0~5시) 평균 잔차」 평균 — 3-라의 날마다 차이를 달마다 모았다


,선형회귀,랜덤포레스트,그래디언트 부스팅
달,,,
1월,+912,+713,+670
2월,+493,"+1,292","+1,814"
3월,"-1,914","+1,051",+716
4월,"-1,877",-31,+204
5월,"-2,035",-826,-969
6월,-646,"-1,122",-651
7월,"+2,819",+595,+667
9월,"+1,967",+285,+751
10월,-330,+725,+793


[표 2] 11달 가운데 0보다 아래인 달 수


,0보다 아래인 달
모델,
선형회귀,6달 / 11달
랜덤포레스트,5달 / 11달
그래디언트 부스팅,4달 / 11달


# 물음 4 — 7단계에 넘길 열을 한 표로 확정한다

> **결정** — 습도는 넣지 않는다(습도 그대로 · 습하면 1 · 불쾌지수 75 넘는 만큼 모두, 선형회귀와 트리 계열 모두). 그 밖의 열은 4-가의 「넣는 것」 표 그대로다(선형회귀의 3시간 전 기온은 적는 방식만 바꾼다 — 아래 같이 정한 것 1)

## 4-가 — 지금 넣는 열과 뺀 열, 한 표

**무엇을 보는 셀인가.** 아래 [표 A]·[표 B]는 지금까지 정한 것을 글로 모은 것이고, 코드 셀은 준비 ②가 모델에 실제로 넣는 열을 묶음마다 셉니다. [표 A]의 열 수와 코드 셀의 [표 1]·[표 2]가 같은지 봅니다. 준비 ①·② 다음에 돌리면 됩니다. 단계 이름 — 4단계는 무엇이 LNG를 움직이는지 본 단계, 5단계는 있는 열로 모델에 넣을 열을 만든 단계입니다.

**[표 A] 넣는 것**

**[표 B] 뺀 것**

In [359]:
# 6단계 분석계획 · 4-가
묶음 = {"시_": "몇 시인가", "월_": "몇 월인가", "날_": "무슨 날인가",                      # 열 이름 앞부분 → 묶음 이름
        "쉬는 날×": "쉬는 날이면서 몇 시인가", "3시간 전": "3시간 전 기온", "일사": "햇빛 세기(일사)"}
def 묶음이름(열):                                                                  # 선형회귀 열 하나가 어느 묶음인지 찾는다
    for 앞, 이름 in 묶음.items():
        if 열.startswith(앞):
            return 이름
    return "기온"                                                                    # 남는 둘 — 16도보다 추운 만큼 · 24도보다 더운 만큼
def 번호(열):                                                                      # 열 이름 속 숫자 — 「시_10」이 「시_9」 뒤에 오게 줄 세울 때 쓴다
    숫자 = "".join(글자 for 글자 in 열 if 글자.isdigit())
    return int(숫자) if 숫자 else 0
선형표 = pd.DataFrame({"열": X선형.columns, "묶음": [묶음이름(c) for c in X선형.columns]})   # 준비 ②가 실제로 넣는 열과 그 묶음
선형표 = 선형표.assign(번호=선형표["열"].map(번호)).sort_values("번호", kind="stable")     # 1시·2시 … 23시 순서로
순서 = ["몇 시인가", "몇 월인가", "무슨 날인가", "기온", "3시간 전 기온", "햇빛 세기(일사)", "쉬는 날이면서 몇 시인가"]  # [표 A]와 같은 순서
표1 = 선형표.groupby("묶음").agg(열수=("열", "size"), 첫열=("열", "first"), 끝열=("열", "last")).reindex(순서)  # 묶음마다 열 수와 첫·끝 열 이름
print(f"[표 1] 선형회귀에 넣는 열 — 묶음마다 몇 개인가 (모두 {X선형.shape[1]}개)")
display(표1.rename(columns={"열수": "열 수", "첫열": "첫 열", "끝열": "끝 열"}))

트리뜻 = {"hour24": "몇 시인가(0~23)", "month": "몇 월인가(1~12)", "icsr": "햇빛 세기(일사)",     # 트리 계열 열 이름 → 뜻
          "ta": "기온 그대로", "dow": "무슨 요일인가(0 = 월요일 ~ 6 = 일요일)",
          "is_holiday": "공휴일이면 1", "명절": "명절이면 1"}
print(f"[표 2] 트리 계열에 넣는 열 (모두 {X트리.shape[1]}개)")
display(pd.DataFrame({"열 이름": X트리.columns, "무엇": [트리뜻[c] for c in X트리.columns]}).set_index("열 이름"))

[표 1] 선형회귀에 넣는 열 — 묶음마다 몇 개인가 (모두 67개)


,열 수,첫 열,끝 열
묶음,,,
몇 시인가,23,시_1,시_23
몇 월인가,11,월_2,월_12
무슨 날인가,5,날_공휴일,날_화~금
기온,2,16도보다 추운 만큼,24도보다 더운 만큼
3시간 전 기온,2,3시간 전 16도보다 추운 만큼,3시간 전 24도보다 더운 만큼
햇빛 세기(일사),1,일사,일사
쉬는 날이면서 몇 시인가,23,쉬는 날×1시,쉬는 날×23시


[표 2] 트리 계열에 넣는 열 (모두 7개)


,무엇
열 이름,
hour24,몇 시인가(0~23)
month,몇 월인가(1~12)
icsr,햇빛 세기(일사)
ta,기온 그대로
dow,무슨 요일인가(0 = 월요일 ~ 6 = 일요일)
is_holiday,공휴일이면 1
명절,명절이면 1


## 4-다 — 지금 모델에 습도를 더하면 기준 셋을 넘나

**무엇을 보는 셀인가.** 지금 모델(선형회귀 67열, 트리 계열 7열)에 습도를 넣는 방식을 하나씩 더해 봅니다. 준비 ①·② 다음에 돌리면 됩니다.

| 표 | 무엇이 나오나 | 어느 기준으로 읽나 |
|---|---|---|
| [표 1] | 선형회귀 — 두 해를 배운 R², 2023년으로 배워 2024년을 맞힌 R², 지금 모델과의 차이. 줄은 지금 모델과, 습도 그대로 · 습하면 1 · 불쾌지수 75 넘는 만큼을 하나씩 더한 것 | 기준 1(「차이 (2024년)」이 +0.002를 넘나) · 기준 2(「차이 (두 해)」만큼이 「차이 (2024년)」에서도 나오나) |
| [표 2] | 트리 계열 — 습도 그대로를 더하기 전과 뒤의, 2023년으로 배워 2024년을 맞힌 R²와 차이 | 기준 1 |
| [표 3] | 선형회귀에 더한 열의 VIF | 기준 3(10 아래인가) |

In [360]:
# 6단계 분석계획 · 4-다
가운데 = 학습.loc[배움, "hm"].median()                                               # 2023년 시각들의 습도 가운데값 — 「습하면 1」의 선
불쾌지수 = 9 / 5 * 학습["ta"] - 0.55 * (1 - 학습["hm"] / 100) * (9 / 5 * 학습["ta"] - 26) + 32   # 기상청 식 — 5단계 7-나와 같다
더할열 = {"습도 그대로": 학습["hm"],                                                   # 서울 108 지점 상대습도(%) 한 열
          "습하면 1": (학습["hm"] > 가운데).astype(float),                              # 2023년 가운데값보다 습하면 1, 아니면 0
          "불쾌지수 75 넘는 만큼": (불쾌지수 - 75).clip(lower=0)}                        # 75를 넘는 만큼 — 넘지 않으면 0

def 선형R2(X):                                                                     # 선형회귀의 R² 둘
    두해 = LinearRegression().fit(X, y).score(X, y)                                  # 두 해를 배워 두 해를 맞힌 R²
    맞힘R2 = LinearRegression().fit(X[배움], y[배움]).score(X[맞힘], y[맞힘])           # 2023년으로 배워 2024년을 맞힌 R²
    return 두해, 맞힘R2

def VIF(X, 열):                                                                     # 그 열을 나머지 열로 맞힌 R²로 계산한다 — VIF = 1 / (1 − R²)
    나머지 = X.drop(columns=열)
    맞힌정도 = LinearRegression().fit(나머지, X[열]).score(나머지, X[열])
    return 1 / (1 - 맞힌정도)

기준 = 선형R2(X선형)                                                                 # 지금 모델(67열)의 R² 둘
줄 = {"지금 모델 (67열)": 기준}
vif = {}
for 이름, 열 in 더할열.items():
    X = X선형.assign(**{이름: 열})                                                    # 지금 모델에 한 열을 더한다
    줄["+ " + 이름] = 선형R2(X)
    vif[이름] = VIF(X, 이름)
표1 = pd.DataFrame(줄, index=["R² (두 해)", "R² (2024년 맞히기)"]).T                  # 줄마다 R² 둘
표1.insert(1, "차이 (두 해)", (표1["R² (두 해)"] - 기준[0]).map("{:+.4f}".format))    # 지금 모델보다 얼마나 오르나
표1["차이 (2024년)"] = (표1["R² (2024년 맞히기)"] - 기준[1]).map("{:+.4f}".format)
표1[["R² (두 해)", "R² (2024년 맞히기)"]] = 표1[["R² (두 해)", "R² (2024년 맞히기)"]].apply(lambda s: s.map("{:.4f}".format))
print("[표 1] 선형회귀 R² — 「두 해」는 두 해를 배워 두 해를 맞힘, 「2024년 맞히기」는 2023년으로 배워 2024년을 맞힘")
display(표1)

def 트리R2(X):                                                                     # 트리 계열 둘 — 2023년으로 배워 2024년을 맞힌 R²
    숲 = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0).fit(X[배움], y[배움])
    부스팅 = HistGradientBoostingRegressor(early_stopping=False, random_state=0).fit(X[배움], y[배움])
    return 숲.score(X[맞힘], y[맞힘]), 부스팅.score(X[맞힘], y[맞힘])
트리기준 = 트리R2(X트리)                                                               # 지금 모델(7열)
트리더함 = 트리R2(X트리.assign(**{"습도 그대로": 학습["hm"]}))                            # 습도 그대로 한 열을 더한다
표2 = pd.DataFrame({"지금 모델 (7열)": 트리기준, "+ 습도 그대로": 트리더함},
                   index=["랜덤포레스트", "그래디언트 부스팅"])
표2["차이 (2024년)"] = (표2["+ 습도 그대로"] - 표2["지금 모델 (7열)"]).map("{:+.4f}".format)
표2[["지금 모델 (7열)", "+ 습도 그대로"]] = 표2[["지금 모델 (7열)", "+ 습도 그대로"]].apply(lambda s: s.map("{:.4f}".format))
print("[표 2] 트리 계열 R² — 2023년으로 배워 2024년을 맞힘")
display(표2)

print(f"[표 3] 선형회귀에 더한 열의 VIF — 10을 넘으면 다른 열과 겹친다고 본다 (「습하면 1」의 선은 2023년 습도 가운데값 {가운데:.0f}%)")
display(pd.Series(vif).map("{:.1f}".format).rename("VIF").to_frame())

[표 1] 선형회귀 R² — 「두 해」는 두 해를 배워 두 해를 맞힘, 「2024년 맞히기」는 2023년으로 배워 2024년을 맞힘


,R² (두 해),차이 (두 해),R² (2024년 맞히기),차이 (2024년)
지금 모델 (67열),0.8120,+0.0000,0.7524,+0.0000
+ 습도 그대로,0.8131,+0.0011,0.7493,-0.0032
+ 습하면 1,0.8126,+0.0006,0.7507,-0.0017
+ 불쾌지수 75 넘는 만큼,0.8133,+0.0013,0.7536,+0.0012


[표 2] 트리 계열 R² — 2023년으로 배워 2024년을 맞힘


,지금 모델 (7열),+ 습도 그대로,차이 (2024년)
랜덤포레스트,0.7618,0.7618,+0.0000
그래디언트 부스팅,0.7807,0.7816,+0.0009


[표 3] 선형회귀에 더한 열의 VIF — 10을 넘으면 다른 열과 겹친다고 본다 (「습하면 1」의 선은 2023년 습도 가운데값 68%)


,VIF
습도 그대로,2.1
습하면 1,1.6
불쾌지수 75 넘는 만큼,8.9


## 4-라 — 지금 기온과 3시간 전 기온이 겹치는 것을 어떻게 둘까

**무엇을 보는 셀인가.** 3시간 전 기온 두 열을 그대로 넣는 지금 방식과, 그 두 열 대신 「3시간 동안 바뀐 만큼」(지금 값 − 3시간 전 값)을 넣는 방식을 나란히 봅니다. 예를 들어 지금 16도보다 4도 추운데 3시간 전에는 6도 추웠다면, 「3시간 동안 바뀐 만큼 (추운 쪽)」은 4 − 6 = −2입니다(3시간 사이에 2도만큼 덜 추워졌다). 두 방식은 같은 정보를 다르게 적은 것이라 선형회귀의 예측은 같아야 합니다. 준비 ①·② 다음에 돌리면 됩니다.

| 표 | 무엇이 나오나 |
|---|---|
| [표 1] | 기온 열 넷의 VIF — 왼쪽은 지금 방식, 오른쪽은 「바뀐 만큼」 방식. 같은 줄의 두 열은 서로 바꾼 열이다 |
| [표 2] | 두 방식의 결정계수(2023년으로 배워 2024년을 맞힘)와, 2024년 시각마다 예측이 가장 크게 다른 양 |

In [361]:
# 6단계 분석계획 · 4-라
def VIF값(X, 열):                                                                   # 그 열을 나머지 열로 맞힌 R²로 계산한다 — VIF = 1 / (1 − R²)
    나머지 = X.drop(columns=열)
    return 1 / (1 - LinearRegression().fit(나머지, X[열]).score(나머지, X[열]))

지금열 = ["16도보다 추운 만큼", "24도보다 더운 만큼", "3시간 전 16도보다 추운 만큼", "3시간 전 24도보다 더운 만큼"]
바뀐열 = ["16도보다 추운 만큼", "24도보다 더운 만큼", "3시간 동안 바뀐 만큼 (추운 쪽)", "3시간 동안 바뀐 만큼 (더운 쪽)"]
X바뀐 = X선형.assign(**{"3시간 동안 바뀐 만큼 (추운 쪽)": X선형["16도보다 추운 만큼"] - X선형["3시간 전 16도보다 추운 만큼"],   # 지금 값 − 3시간 전 값
                       "3시간 동안 바뀐 만큼 (더운 쪽)": X선형["24도보다 더운 만큼"] - X선형["3시간 전 24도보다 더운 만큼"]}
                      ).drop(columns=지금열[2:])                                     # 3시간 전 두 열 대신 「바뀐 만큼」 두 열

표1 = pd.DataFrame({"지금 방식 — 열": 지금열, "VIF (지금)": [VIF값(X선형, c) for c in 지금열],
                    "「바뀐 만큼」 방식 — 열": 바뀐열, "VIF (바뀐 만큼)": [VIF값(X바뀐, c) for c in 바뀐열]})
표1[["VIF (지금)", "VIF (바뀐 만큼)"]] = 표1[["VIF (지금)", "VIF (바뀐 만큼)"]].apply(lambda s: s.map("{:.1f}".format))
print("[표 1] 기온 열 넷의 VIF — 10을 넘으면 다른 열과 겹친다고 본다. 같은 줄의 두 열은 서로 바꾼 열이다")
display(표1.set_index("지금 방식 — 열"))

모델지금 = LinearRegression().fit(X선형[배움], y[배움])                               # 두 방식 모두 2023년으로 배운다
모델바뀐 = LinearRegression().fit(X바뀐[배움], y[배움])
예측지금 = 모델지금.predict(X선형[맞힘]); 예측바뀐 = 모델바뀐.predict(X바뀐[맞힘])          # 2024년 시각마다 예측
표2 = pd.DataFrame({"2024년 맞히기 R²": [모델지금.score(X선형[맞힘], y[맞힘]), 모델바뀐.score(X바뀐[맞힘], y[맞힘])]},
                   index=["지금 방식", "「바뀐 만큼」 방식"]).round(4)
print("[표 2] 두 방식의 결정계수 — 2023년으로 배워 2024년을 맞힘")
display(표2)
print(f"2024년 {int(맞힘.sum()):,}시각에서 두 방식의 예측이 가장 크게 다른 양: {abs(예측지금 - 예측바뀐).max():.6f} MWh")

[표 1] 기온 열 넷의 VIF — 10을 넘으면 다른 열과 겹친다고 본다. 같은 줄의 두 열은 서로 바꾼 열이다


,VIF (지금),「바뀐 만큼」 방식 — 열,VIF (바뀐 만큼)
지금 방식 — 열,,,
16도보다 추운 만큼,35.5,16도보다 추운 만큼,4.7
24도보다 더운 만큼,6.4,24도보다 더운 만큼,2.2
3시간 전 16도보다 추운 만큼,35.8,3시간 동안 바뀐 만큼 (추운 쪽),2.0
3시간 전 24도보다 더운 만큼,5.7,3시간 동안 바뀐 만큼 (더운 쪽),1.7


[표 2] 두 방식의 결정계수 — 2023년으로 배워 2024년을 맞힘


,2024년 맞히기 R²
지금 방식,0.7524
「바뀐 만큼」 방식,0.7524


2024년 8,010시각에서 두 방식의 예측이 가장 크게 다른 양: 0.000000 MWh


## 4-마 — 트리 계열에도 3시간 전 기온을 줄까

**무엇을 보는 셀인가.** 트리 계열 일곱 열에 「3시간 전 기온 그대로」 한 열을 더하기 전과 뒤의 결정계수(2023년으로 배워 2024년을 맞힘)입니다. 5단계 4-마에서 비슷한 것을 봤을 때는 랜덤포레스트는 오르고 그래디언트 부스팅은 내려가 두 모델이 갈렸는데, 그때 트리 계열 열에는 요일·공휴일·명절이 없었습니다. 지금 일곱 열로 다시 봅니다. 준비 ①·② 다음에 돌리면 됩니다.

In [362]:
# 6단계 분석계획 · 4-마
def 트리두R2(X):                                                                    # 트리 계열 둘 — 2023년으로 배워 2024년을 맞힌 R²
    숲 = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0).fit(X[배움], y[배움])
    부스팅 = HistGradientBoostingRegressor(early_stopping=False, random_state=0).fit(X[배움], y[배움])
    return 숲.score(X[맞힘], y[맞힘]), 부스팅.score(X[맞힘], y[맞힘])

전 = 트리두R2(X트리)                                                                  # 지금 일곱 열
후 = 트리두R2(X트리.assign(**{"3시간 전 기온": 학습["3시간 전 기온"]}))                     # 3시간 전 기온 그대로 한 열을 더한다
표 = pd.DataFrame({"지금 (7열)": 전, "+ 3시간 전 기온": 후}, index=["랜덤포레스트", "그래디언트 부스팅"])
표["차이 (2024년)"] = (표["+ 3시간 전 기온"] - 표["지금 (7열)"]).map("{:+.4f}".format)
표[["지금 (7열)", "+ 3시간 전 기온"]] = 표[["지금 (7열)", "+ 3시간 전 기온"]].apply(lambda s: s.map("{:.4f}".format))
print("[표 1] 트리 계열 R² — 2023년으로 배워 2024년을 맞힘. 3시간 전 기온 그대로 한 열을 더하기 전과 뒤")
display(표)

[표 1] 트리 계열 R² — 2023년으로 배워 2024년을 맞힘. 3시간 전 기온 그대로 한 열을 더하기 전과 뒤


,지금 (7열),+ 3시간 전 기온,차이 (2024년)
랜덤포레스트,0.7618,0.7629,+0.0011
그래디언트 부스팅,0.7807,0.7816,+0.0009


# 물음 5 — 무엇을 맞히고, 기간을 어떻게 나누고, 시험을 어떻게 여러 번 할까

> **결정** — 시간별 LNG 발전량(MWh)을 맞히고, 2023·2024년으로 배워 2025년에 한 번 채점하고 2026년 1~6월과 비교한다. 6단계의 시험은 여러 번 나누지 않고 지금까지처럼 「2023년으로 배워 2024년 맞히기」 한 번으로 한다.

## 5-나 — 시험을 여러 번 나눠 하려면 무엇이 맞아야 하고, 어느 방법으로 할까

**무엇을 보는 셀인가.** 아래 후보 셋이 실제 시각으로 어떻게 나뉘는지를 표 셋으로 보여 줍니다 — 시험마다 배우는 달과 시각 수, 맞히는 달과 시각 수입니다. 결정계수는 내지 않습니다. 여기서 만든 나누는 함수(`시험나누기`)는 5-다부터 그대로 씁니다.

**왜 여러 번 하나.** 열 하나를 넣었더니 2024년을 맞힌 결정계수가 0.001 올랐다면, 그것이 그 열 덕분인지 2024년이 우연히 그렇게 나온 것인지 시험 한 번으로는 구별할 수 없습니다. 시험을 여러 번 하면 오른 것이 시험마다 같은 쪽으로 나오는지 볼 수 있습니다. 여기서 정한 방법은 5-바(0.002 확인)와 물음 6(모델과 하이퍼파라미터 고르기)에서 씁니다.

**이미 정해진 것 셋** — 어느 방법이든 지킵니다.

1. **시간 순서** — 배우는 시각은 맞히는 시각보다 앞서야 합니다. 시각을 무작위로 섞어 나누면 같은 날 앞뒤 시각이 배우는 쪽과 맞히는 쪽에 나뉘어 들어가는데, 이어진 시각끼리는 값이 비슷해서 성적이 실제보다 좋게 나옵니다.
2. **배우는 쪽에 열두 달이 다 들어간다** — 1~6월로 배우고 7월을 맞히면 여름을 한 번도 못 본 채로 여름을 맞히게 됩니다. 그래서 2023년 한 해는 늘 배우는 쪽에 두고, 맞히는 시험은 2024년 안에서만 합니다.
3. **2025년은 쓰지 않는다** — 2025년 LNG는 7단계 마지막 채점에 한 번만 씁니다.

**후보 셋**

| 방법 | 어떻게 나누나 | 시험 수 | 시험 하나가 맞히는 기간 |
|---|---|---|---|
| 지금처럼 한 번 | 2023년 한 해로 배워 2024년 한 해를 맞힌다 | 1번 | 한 해(8월 빼고 11개 달) |
| 한 달씩 넓혀 가며 | 2024년의 달마다, 2023년 1월부터 그 달 바로 전까지로 배우고 그 달을 맞힌다 | 11번 | 한 달 |
| 석 달씩 넓혀 가며 | 2024년을 석 달씩 넷으로 나눠, 그 석 달 바로 전까지로 배우고 그 석 달을 맞힌다 | 4번 | 석 달(7~9월은 8월이 빠져 7·9월) |

- 예: 「한 달씩 넓혀 가며」의 다섯째 시험은 2023년 1월 ~ 2024년 4월로 배우고 2024년 5월을 맞힙니다. 뒤 시험으로 갈수록 배우는 기간이 늘어서, 마지막 시험은 7단계처럼 두 해 가까이 배웁니다.
- 7단계는 두 해로 배워 그다음 한 해(2025년)를 맞힙니다. 여러 번 나누는 방법은 바로 다음 한 달이나 석 달을 맞혀서, 7단계보다 가까운 앞날을 맞힙니다. 그래서 이 시험으로 7단계 성적을 미리 알 수는 없고, 열이나 모델을 바꿨을 때 나아지는지 비교하는 데 씁니다.

In [363]:
# 6단계 분석계획 · 5-나
월 = 학습.index.to_period("M")                                       # 시각마다 몇 년 몇 월인지 — 「2024-05」 같은 꼴
맞힐달 = sorted(월[np.asarray(맞힘)].unique())                         # 2024년에 자료가 있는 달 — 8월은 3단계에서 빼서 11개

def 시험나누기(방법):                                                 # 방법마다 (배우는 시각, 맞히는 시각) 짝을 시험 수만큼 돌려준다
    if 방법 == "지금처럼 한 번":
        return [(np.asarray(배움), np.asarray(맞힘))]                    # 준비 ②와 같다 — 2023년으로 배워 2024년을 맞힌다
    if 방법 == "한 달씩 넓혀 가며":
        묶음 = [[m] for m in 맞힐달]                                     # 시험 하나가 2024년의 한 달을 맞힌다
    elif 방법 == "석 달씩 넓혀 가며":
        묶음 = [[m for m in 맞힐달 if (m.month - 1) // 3 == q] for q in range(4)]   # 1~3월 · 4~6월 · 7~9월 · 10~12월
    시험 = []
    for 달들 in 묶음:
        배우는칸 = np.asarray(월 < min(달들))                             # 맞히는 첫 달 바로 전까지의 모든 시각 — 늘 2023년 1월부터
        맞히는칸 = np.asarray(월.isin(달들))                              # 맞히는 달들의 모든 시각
        시험.append((배우는칸, 맞히는칸))
    return 시험

def 달글(칸):                                                         # 칸이 가리키는 달을 「2023년 1~12월 · 2024년 1~7·9월」처럼 적는다
    해마다 = {}
    for m in sorted(월[칸].unique()):
        해마다.setdefault(m.year, []).append(m.month)                   # 해마다 달 번호를 모은다
    글 = []
    for 해, 달들 in 해마다.items():
        토막 = [[달들[0]]]
        for 달 in 달들[1:]:
            if 달 == 토막[-1][-1] + 1:
                토막[-1].append(달)                                      # 바로 이어지는 달이면 같은 토막에 붙인다
            else:
                토막.append([달])                                        # 끊기면(8월이 빠진 곳) 새 토막을 연다
        글.append(f"{해}년 " + "·".join(f"{t[0]}~{t[-1]}" if len(t) >= 3 else "·".join(map(str, t)) for t in 토막) + "월")
    return " · ".join(글)

for 번호, 방법 in enumerate(["지금처럼 한 번", "한 달씩 넓혀 가며", "석 달씩 넓혀 가며"], start=1):
    줄 = []
    for i, (배우는칸, 맞히는칸) in enumerate(시험나누기(방법), start=1):
        줄.append({"시험": i,
                   "배우는 달": 달글(배우는칸), "배우는 시각 수": f"{int(배우는칸.sum()):,}",
                   "맞히는 달": 달글(맞히는칸), "맞히는 시각 수": f"{int(맞히는칸.sum()):,}"})
    print(f"[표 {번호}] {방법} — 시험 {len(줄)}번")              # 표 제목 — 방법 이름과 시험 수
    display(pd.DataFrame(줄).set_index("시험"))                   # 시험마다 한 줄 — 결정계수는 없고 나누는 것만

[표 1] 지금처럼 한 번 — 시험 1번


,배우는 달,배우는 시각 수,맞히는 달,맞히는 시각 수
시험,,,,
1,2023년 1~12월,"8,757",2024년 1~7·9~12월,"8,010"


[표 2] 한 달씩 넓혀 가며 — 시험 11번


,배우는 달,배우는 시각 수,맞히는 달,맞히는 시각 수
시험,,,,
1,2023년 1~12월,"8,757",2024년 1월,744
2,2023년 1~12월 · 2024년 1월,"9,501",2024년 2월,672
3,2023년 1~12월 · 2024년 1·2월,"10,173",2024년 3월,741
4,2023년 1~12월 · 2024년 1~3월,"10,914",2024년 4월,720
5,2023년 1~12월 · 2024년 1~4월,"11,634",2024년 5월,744
6,2023년 1~12월 · 2024년 1~5월,"12,378",2024년 6월,720
7,2023년 1~12월 · 2024년 1~6월,"13,098",2024년 7월,744
8,2023년 1~12월 · 2024년 1~7월,"13,842",2024년 9월,717
9,2023년 1~12월 · 2024년 1~7·9월,"14,559",2024년 10월,744


[표 3] 석 달씩 넓혀 가며 — 시험 4번


,배우는 달,배우는 시각 수,맞히는 달,맞히는 시각 수
시험,,,,
1,2023년 1~12월,"8,757",2024년 1~3월,"2,157"
2,2023년 1~12월 · 2024년 1~3월,"10,914",2024년 4~6월,"2,184"
3,2023년 1~12월 · 2024년 1~6월,"13,098",2024년 7·9월,"1,461"
4,2023년 1~12월 · 2024년 1~7·9월,"14,559",2024년 10~12월,"2,208"


## 5-다 — 방법마다 지금 모델 셋을 돌리면 무엇이 나오나

**무엇을 보는 셀인가.** 지금 모델 셋(선형회귀 · 랜덤포레스트 · 그래디언트 부스팅)을 방법마다 돌립니다. [표 1]은 2024년 전체로 모은 결정계수, [표 2]는 달마다 결정계수(왼쪽은 2023년으로 배운 모델 하나를 달마다 나눠 본 것, 오른쪽은 달마다 다시 배운 것), [표 3]은 석 달마다 결정계수입니다. 표마다 그 기간에 결정계수가 가장 높았던 모델을 같이 적었습니다.

- **점수가 높게 나온 방법이 좋은 방법은 아닙니다.** 배우는 자료가 많고 맞히는 달에 가까울수록 점수가 높게 나오는 것이 당연해서, 높은 쪽을 고르면 실제보다 좋게 보는 쪽을 고르게 됩니다. 이 셀은 방법을 바꾸면 결론(어느 모델이 더 잘 맞히나)이 달라지는지 보려는 것입니다.
- 한 달 안에서 낸 결정계수는 그 달 안의 오르내림을 기준으로 내서, 한 해로 낸 것보다 낮게 나오기 쉽습니다.
- 「지금처럼 한 번」의 [표 1] 숫자는 물음 4의 4-라·4-마와 같은 계산입니다. 물음 2의 숫자와 조금 다른 것은, 물음 2가 2024년 12월 30·31일을 빼고 냈기 때문입니다(「52주 전 같은 요일」 기준선을 낼 수 없는 날이라 일곱 방법 모두에서 뺐습니다).
- 랜덤포레스트를 열여섯 번 배워서 30초쯤 걸립니다. 5-나 코드 셀을 먼저 돌려야 합니다(`시험나누기`를 씁니다).

In [364]:
# 6단계 분석계획 · 5-다
모델들 = {"선형회귀": (lambda: LinearRegression(), X선형),                                         # 물음 2·4와 같은 세 모델
          "랜덤포레스트": (lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0), X트리),
          "그래디언트 부스팅": (lambda: HistGradientBoostingRegressor(early_stopping=False, random_state=0), X트리)}
방법들 = ["지금처럼 한 번", "한 달씩 넓혀 가며", "석 달씩 넓혀 가며"]                               # 5-나의 세 방법
맞힘칸 = np.asarray(맞힘)                                                                          # 2024년 시각
실제 = y[맞힘칸]                                                                                   # 2024년 실제 LNG(MWh)
월24 = 월[맞힘칸]                                                                                  # 2024년 시각마다 몇 월인지

def 시험돌리기(방법, 만들기, X):                                                                   # 방법대로 배우고 맞혀서, 2024년 시각마다 예측 하나를 돌려준다
    예측 = pd.Series(np.nan, index=y.index)
    for 배우는칸, 맞히는칸 in 시험나누기(방법):
        예측[맞히는칸] = 만들기().fit(X[배우는칸], y[배우는칸]).predict(X[맞히는칸])               # 그 시험의 배우는 시각으로만 배운다
    return 예측[맞힘칸]

def R2(실, 예):                                                                                     # 결정계수 — 지금까지와 같은 식
    return 1 - ((실 - 예) ** 2).sum() / ((실 - 실.mean()) ** 2).sum()

예측5다 = {(방법, 이름): 시험돌리기(방법, 만들기, X)                                                # 방법 셋 × 모델 셋 — 5-라에서도 쓴다
           for 방법 in 방법들 for 이름, (만들기, X) in 모델들.items()}

def 나눠보기(방법, 묶음):                                                                          # 묶음(한 달 또는 석 달)마다 세 모델의 결정계수와 그때 가장 높은 모델
    줄 = {}
    for 이름표, 칸 in 묶음:
        값 = {이름: R2(실제[칸], 예측5다[(방법, 이름)][칸]) for 이름 in 모델들}
        줄[이름표] = {**{이름: f"{v:.3f}" for 이름, v in 값.items()}, "가장 높은 모델": max(값, key=값.get)}
    return pd.DataFrame(줄).T.rename_axis("맞힌 기간")

달묶음 = [(f"2024년 {m.month}월", np.asarray(월24 == m)) for m in 맞힐달]                              # 8월이 빠진 11개 달
석달 = [[m for m in 맞힐달 if (m.month - 1) // 3 == q] for q in range(4)]                            # 1~3월 · 4~6월 · 7·9월 · 10~12월
석달묶음 = [(달글(np.asarray(월.isin(달들))), np.asarray(월24.isin(달들))) for 달들 in 석달]

표1 = pd.DataFrame({방법: {이름: f"{R2(실제, 예측5다[(방법, 이름)]):.4f}" for 이름 in 모델들} for 방법 in 방법들})
표2 = pd.concat({"지금처럼 한 번 — 모델 하나를 달마다 나눠 봄": 나눠보기("지금처럼 한 번", 달묶음),
                 "한 달씩 넓혀 가며 — 달마다 다시 배움": 나눠보기("한 달씩 넓혀 가며", 달묶음)}, axis=1)   # 같은 달을 두 방법으로 나란히
print("[표 1] 2024년 전체로 모은 결정계수 — 방법마다 (시험마다 맞힌 값을 2024년 한 해로 모아서 냄)")
display(표1)
print("[표 2] 달마다 결정계수 — 왼쪽은 2023년으로 배운 모델 하나로 맞힌 것, 오른쪽은 그 달 바로 전까지로 다시 배워 맞힌 것")
display(표2)
print("[표 3] 석 달씩 넓혀 가며 — 석 달마다 그 석 달 바로 전까지로 다시 배워 맞힌 결정계수")
display(나눠보기("석 달씩 넓혀 가며", 석달묶음))

[표 1] 2024년 전체로 모은 결정계수 — 방법마다 (시험마다 맞힌 값을 2024년 한 해로 모아서 냄)


,지금처럼 한 번,한 달씩 넓혀 가며,석 달씩 넓혀 가며
선형회귀,0.7524,0.7503,0.7488
랜덤포레스트,0.7618,0.7682,0.7659
그래디언트 부스팅,0.7807,0.7847,0.7804


[표 2] 달마다 결정계수 — 왼쪽은 2023년으로 배운 모델 하나로 맞힌 것, 오른쪽은 그 달 바로 전까지로 다시 배워 맞힌 것


지금처럼 한 번 — 모델 하나를 달마다 나눠 봄                              \
                                선형회귀 랜덤포레스트 그래디언트 부스팅   가장 높은 모델   
맞힌 기간                                                              
2024년 1월                       0.775  0.848     0.867  그래디언트 부스팅   
2024년 2월                       0.732  0.673     0.703       선형회귀   
2024년 3월                       0.701  0.714     0.751  그래디언트 부스팅   
2024년 4월                       0.643  0.697     0.719  그래디언트 부스팅   
2024년 5월                       0.596  0.674     0.671     랜덤포레스트   
2024년 6월                       0.727  0.760     0.763  그래디언트 부스팅   
2024년 7월                       0.789  0.832     0.841  그래디언트 부스팅   
2024년 9월                       0.848  0.754     0.823       선형회귀   
2024년 10월                      0.642  0.628     0.634       선형회귀   
2024년 11월                      0.549  0.643     0.598     랜덤포레스트   
2024년 12월                      0.706  0.706     0.735  그래디언트 부스팅   

          한 달씩 넓혀 가며 — 달마다 다시 배움                              
                            선형회귀 랜덤포레스트 그래디언트 부스팅   가장 높은 모델  
맞힌 기간                                                         
2024년 1월                   0.775  0.848     0.867  그래디언트 부스팅  
2024년 2월                   0.737  0.664     0.703       선형회귀  
2024년 3월                   0.701  0.714     0.751  그래디언트 부스팅  
2024년 4월                   0.634  0.692     0.709  그래디언트 부스팅  
2024년 5월                   0.613  0.695     0.688     랜덤포레스트  
2024년 6월                   0.720  0.762     0.774  그래디언트 부스팅  
2024년 7월                   0.784  0.837     0.847  그래디언트 부스팅  
2024년 9월                   0.840  0.766     0.820       선형회귀  
2024년 10월                  0.633  0.635     0.636  그래디언트 부스팅  
2024년 11월                  0.542  0.670     0.622     랜덤포레스트  
2024년 12월                  0.712  0.721     0.745  그래디언트 부스팅

[표 3] 석 달씩 넓혀 가며 — 석 달마다 그 석 달 바로 전까지로 다시 배워 맞힌 결정계수


,선형회귀,랜덤포레스트,그래디언트 부스팅,가장 높은 모델
맞힌 기간,,,,
2024년 1~3월,0.753,0.768,0.794,그래디언트 부스팅
2024년 4~6월,0.671,0.726,0.727,그래디언트 부스팅
2024년 7·9월,0.815,0.804,0.829,그래디언트 부스팅
2024년 10~12월,0.667,0.697,0.699,그래디언트 부스팅


## 5-라 — 석 달씩의 결정계수가 높아 보이는 까닭: 같은 모델로 놓는 기간만 바꿔 보기

**무엇을 보는 셀인가.** [표 1]은 2023년으로 배운 모델 하나의 2024년 예측을 그대로 두고, 한 달씩 놓고 낸 결정계수의 평균 · 석 달씩 놓고 낸 평균 · 한 해를 통째로 놓고 낸 값을 나란히 놓습니다. 모델은 그대로이고 놓는 기간만 다릅니다. [표 2]는 같은 석 달을 「지금처럼 한 번」과 「석 달씩 넓혀 가며」로 맞힌 결정계수입니다 — 놓는 기간을 똑같이 맞춘 비교입니다. 5-다 셀을 먼저 돌려야 합니다(`예측5다`를 씁니다).

In [365]:
# 6단계 분석계획 · 5-라
분기 = {달글(np.asarray(월.isin(달들))): np.asarray(월24.isin(달들)) for 달들 in 석달}             # 석 달 묶음 넷 — 5-다와 같다
줄 = []
for 이름 in 모델들:
    예 = 예측5다[("지금처럼 한 번", 이름)]                                                   # 2023년으로 배운 모델 하나의 2024년 예측 — 이 셀에서는 바꾸지 않는다
    달마다 = [R2(실제[칸], 예[칸]) for _, 칸 in 달묶음]                                       # 같은 예측을 한 달씩 놓고 낸 결정계수 11개
    석달마다 = [R2(실제[칸], 예[칸]) for 칸 in 분기.values()]                                  # 같은 예측을 석 달씩 놓고 낸 결정계수 4개
    줄.append({"모델": 이름,
               "한 달씩 놓고 낸 값의 평균 (11개)": f"{np.mean(달마다):.3f}",
               "석 달씩 놓고 낸 값의 평균 (4개)": f"{np.mean(석달마다):.3f}",
               "한 해를 통째로 놓고 낸 값": f"{R2(실제, 예):.3f}"})
print("[표 1] 같은 예측(2023년으로 배운 모델 하나)을 놓는 기간만 바꿔 낸 결정계수 — 모델은 그대로다")
display(pd.DataFrame(줄).set_index("모델"))

표2 = pd.DataFrame({(이름, 방법): {이름표: f"{R2(실제[칸], 예측5다[(방법, 이름)][칸]):.3f}" for 이름표, 칸 in 분기.items()}
                    for 이름 in 모델들 for 방법 in ["지금처럼 한 번", "석 달씩 넓혀 가며"]}).rename_axis("맞힌 석 달")
print("[표 2] 같은 석 달을 두 방법으로 맞힌 결정계수 — 놓는 기간을 똑같이 석 달로 맞춰 비교한다")
display(표2)

[표 1] 같은 예측(2023년으로 배운 모델 하나)을 놓는 기간만 바꿔 낸 결정계수 — 모델은 그대로다


,한 달씩 놓고 낸 값의 평균 (11개),석 달씩 놓고 낸 값의 평균 (4개),한 해를 통째로 놓고 낸 값
모델,,,
선형회귀,0.701,0.730,0.752
랜덤포레스트,0.721,0.745,0.762
그래디언트 부스팅,0.737,0.763,0.781


[표 2] 같은 석 달을 두 방법으로 맞힌 결정계수 — 놓는 기간을 똑같이 석 달로 맞춰 비교한다


선형회귀              랜덤포레스트            그래디언트 부스팅           
             지금처럼 한 번 석 달씩 넓혀 가며 지금처럼 한 번 석 달씩 넓혀 가며  지금처럼 한 번 석 달씩 넓혀 가며
맞힌 석 달                                                                   
2024년 1~3월      0.753      0.753    0.768      0.768     0.794      0.794
2024년 4~6월      0.675      0.671    0.725      0.726     0.732      0.727
2024년 7·9월      0.822      0.815    0.792      0.804     0.833      0.829
2024년 10~12월    0.670      0.667    0.694      0.697     0.692      0.699

## 5-마 — 그래디언트 부스팅이 높으면 지금까지 한 일은 무용한가: 정한 것을 하나씩 되돌려 보기

**무엇을 보는 셀인가.** 2·4·5·6단계에서 정한 것을 하나씩 되돌려, 2023년으로 배워 2024년을 맞힌 결정계수가 어떻게 바뀌는지 봅니다. [표 1]은 그래디언트 부스팅(일곱 열), [표 2]는 선형회귀(67열)입니다. 줄마다 「정한 곳」에 그것을 정한 단계와 물음을 적었습니다.

- 「지금보다」 칸이 음수면 그것을 되돌렸을 때 덜 맞힌다는 뜻이고, 그만큼이 그 결정이 한 일입니다.
- 그래디언트 부스팅은 컴퓨터마다 0.001 안쪽으로 다르게 나오니, 그보다 작은 차이는 같다고 봅니다.
- 2024년 8월을 뺀 것(3단계)은 되돌릴 자료가 이 표에 없어서 넣지 않았습니다.

In [366]:
# 6단계 분석계획 · 5-마
def 한번R2(만들기, X):                                                                       # 지금처럼 한 번 — 2023년으로 배워 2024년을 맞힌 결정계수(물음 1~4와 같은 시험)
    return R2(실제, 만들기().fit(X[배움], y[배움]).predict(X[맞힘]))

요일만 = 학습.assign(**{"날 종류": 학습["dow"].map({0: "월", 1: "화~금", 2: "화~금", 3: "화~금", 4: "화~금", 5: "토", 6: "일"})})   # 공휴일·명절을 모르고 요일만 아는 표
기온네열 = ["16도보다 추운 만큼", "24도보다 더운 만큼", "3시간 전 16도보다 추운 만큼", "3시간 전 24도보다 더운 만큼"]

트리줄 = [("지금 — 일곱 열", "—", X트리),
          ("일사 대신 태양광 설비용량", "5단계 물음 3", X트리.drop(columns="icsr").assign(설비=학습["solar_cap_mw"])),
          ("공휴일·명절을 뺌 — 요일만", "2단계 수집 · 5단계 물음 5", X트리.drop(columns=["is_holiday", "명절"])),
          ("원전을 더함", "6단계 물음 1", X트리.assign(원전=학습["nuke_mwh"])),
          ("일사를 뺌", "4단계 · 5단계 물음 3", X트리.drop(columns="icsr")),
          ("기온을 뺌", "4단계 · 5단계 물음 1", X트리.drop(columns="ta"))]
선형줄 = [("지금 — 67열", "—", X선형),
          ("기온을 16·24도로 나누지 않고 그대로 (지금·3시간 전)", "5단계 물음 1", X선형.drop(columns=기온네열).assign(기온=학습["ta"], 전기온=학습["3시간 전 기온"])),
          ("3시간 전 기온 두 열을 뺌", "5단계 물음 2", X선형.drop(columns=기온네열[2:])),
          ("일사 대신 태양광 설비용량", "5단계 물음 3", X선형.drop(columns="일사").assign(설비=학습["solar_cap_mw"])),
          ("공휴일·명절을 모르고 요일만", "2단계 수집 · 5단계 물음 5", 선형열(요일만)),
          ("쉬는 날 × 시간대 23열을 뺌", "5단계 물음 7", X선형.drop(columns=[c for c in X선형.columns if c.startswith("쉬는 날×")])),
          ("원전을 더함", "6단계 물음 1", X선형.assign(원전=학습["nuke_mwh"]))]

for 번호, (모델이름, 줄들) in enumerate([("그래디언트 부스팅", 트리줄), ("선형회귀", 선형줄)], start=1):
    만들기 = 모델들[모델이름][0]                                                               # 5-다와 같은 설정
    기준 = 한번R2(만들기, 줄들[0][2])                                                           # 지금 열로 낸 값
    표 = []
    for 바꾼것, 정한곳, X in 줄들:
        값 = 기준 if X is 줄들[0][2] else 한번R2(만들기, X)
        표.append({"바꾼 것": 바꾼것, "정한 곳": 정한곳, "2024년 결정계수": f"{값:.4f}", "지금보다": "—" if X is 줄들[0][2] else f"{값 - 기준:+.4f}"})
    print(f"[표 {번호}] {모델이름} — 앞 단계에서 정한 것을 하나씩 되돌리면 2023년으로 배워 2024년을 맞힌 결정계수가 어떻게 되나")
    display(pd.DataFrame(표).set_index("바꾼 것"))

[표 1] 그래디언트 부스팅 — 앞 단계에서 정한 것을 하나씩 되돌리면 2023년으로 배워 2024년을 맞힌 결정계수가 어떻게 되나


,정한 곳,2024년 결정계수,지금보다
바꾼 것,,,
지금 — 일곱 열,—,0.7807,—
일사 대신 태양광 설비용량,5단계 물음 3,0.6246,-0.1561
공휴일·명절을 뺌 — 요일만,2단계 수집 · 5단계 물음 5,0.7283,-0.0524
원전을 더함,6단계 물음 1,0.7480,-0.0327
일사를 뺌,4단계 · 5단계 물음 3,0.7353,-0.0454
기온을 뺌,4단계 · 5단계 물음 1,0.7239,-0.0568


[표 2] 선형회귀 — 앞 단계에서 정한 것을 하나씩 되돌리면 2023년으로 배워 2024년을 맞힌 결정계수가 어떻게 되나


,정한 곳,2024년 결정계수,지금보다
바꾼 것,,,
지금 — 67열,—,0.7524,—
기온을 16·24도로 나누지 않고 그대로 (지금·3시간 전),5단계 물음 1,0.6730,-0.0794
3시간 전 기온 두 열을 뺌,5단계 물음 2,0.7508,-0.0016
일사 대신 태양광 설비용량,5단계 물음 3,0.6539,-0.0985
공휴일·명절을 모르고 요일만,2단계 수집 · 5단계 물음 5,0.7056,-0.0468
쉬는 날 × 시간대 23열을 뺌,5단계 물음 7,0.6515,-0.1009
원전을 더함,6단계 물음 1,0.7267,-0.0257


# 물음 6 — 세 모델 말고 7단계에서 같이 비교할 모델이 더 있나

> **결정** — 7단계에서는 기준선과 모델 넷(선형회귀 · 기온 × 시간대 선형회귀 · 랜덤포레스트 · 그래디언트 부스팅)을 2025년 채점에서 같은 표에 놓는다. 랜덤포레스트는 2024년으로 고른 설정을 쓰고, 그래디언트 부스팅은 지금 설정 그대로다.

## 6-나 — 후보 다섯을 2023년으로 배워 2024년을 맞히면

| 후보 | 만든 방법 |
|---|---|
| 기온 × 시간대 선형회귀 | 지금 기온 두 열(16도보다 추운 만큼 · 24도보다 더운 만큼) 대신, 그 두 열을 시간대마다 나눈 48열을 넣는다 |
| 릿지 · 라쏘 | 지금 선형회귀 열을 표준화하고, 벌점 크기는 2023년 안에서 시간 순서로 나눠 고른다 |
| 굽은 선 선형회귀 | 지금 기온 두 열과 일사 대신, 기온과 일사를 굽은 선으로 나타내는 열을 넣는다 |
| 달마다 따로 | 2023년 그 달로만 배워 2024년 그 달을 맞힌다. 선형회귀는 「몇 월인가」 열을 뺀다 |
| 시간대별 기온을 더한 그래디언트 부스팅 | 트리 계열 일곱 열에 위의 48열을 더한다 |

**무엇을 보는 셀인가** — [표 1]은 2024년 한 해로 모은 결정계수와 비교 대상과의 차이, [표 2]는 11달 가운데 몇 달 나아졌는지(참고)입니다. 비교 대상의 값은 5-다 「지금처럼 한 번」과 같습니다.

In [367]:
# 6단계 분석계획 · 6-나
from sklearn.linear_model import RidgeCV, LassoCV                                    # 기울기에 벌점을 주는 선형회귀 두 가지 — 벌점 크기를 스스로 고른다
from sklearn.preprocessing import StandardScaler, SplineTransformer                  # 표준화 · 굽은 선 열 만들기
from sklearn.pipeline import make_pipeline                                           # 표준화 → 모델을 한 줄로 잇는다
from sklearn.model_selection import TimeSeriesSplit                                  # 벌점 크기를 2023년 안에서 시간 순서로 나눠 고른다

선형 = 모델들["선형회귀"][0]                                                           # 5-다와 같은 선형회귀(비교 대상)
부스팅 = 모델들["그래디언트 부스팅"][0]                                                  # 5-다와 같은 그래디언트 부스팅(비교 대상)
기준예측 = {이름: 예측5다[("지금처럼 한 번", 이름)] for 이름 in ["선형회귀", "그래디언트 부스팅"]}  # 비교 대상의 2024년 예측 — 5-다에서 낸 것 그대로

def 한번(만들기, X):                                                                  # 지금처럼 한 번 — 2023년으로 배워 2024년을 맞힌다
    return pd.Series(만들기().fit(X[배움], y[배움]).predict(X[맞힘]), index=실제.index)

def 달마다따로(만들기, X):                                                            # 1월은 2023년 1월로만 배워 2024년 1월을 맞힌다 — 11달 각각
    예측 = pd.Series(np.nan, index=실제.index)
    for m in sorted(학습.loc[맞힘, "month"].unique()):                                  # 2024년에 있는 달(8월 빼고 11달)
        배칸 = 배움 & (학습["month"] == m)                                              # 2023년 그 달
        맞칸 = 맞힘 & (학습["month"] == m)                                              # 2024년 그 달
        예측[맞칸[맞힘].values] = 만들기().fit(X[배칸], y[배칸]).predict(X[맞칸])
    return 예측

시 = 학습["hour24"]                                                                   # 몇 시인지
기온두열 = ["16도보다 추운 만큼", "24도보다 더운 만큼"]                                     # 선형회귀의 지금 기온 두 열
시간대기온 = pd.DataFrame({f"{c}×{h}시": 학습[c] * (시 == h)                             # 5단계 물음 4의 48열 — 그 시간대일 때만 값, 아니면 0
                           for c in 기온두열 for h in range(24)}, index=학습.index)
굽은선 = SplineTransformer(n_knots=6, degree=3).fit(학습.loc[배움, ["ta", "icsr"]])        # 기온·일사를 굽은 선 열로 — 매듭은 2023년 값으로만 정한다
굽은열 = pd.DataFrame(굽은선.transform(학습[["ta", "icsr"]]), index=학습.index).add_prefix("굽은 선 ")
시간순 = TimeSeriesSplit(n_splits=4)                                                  # 2023년을 시간 순서로 넷으로 나눈다 — 벌점 크기는 2023년 안에서만 고른다
릿지 = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 13), cv=시간순))
라쏘 = lambda: make_pipeline(StandardScaler(), LassoCV(cv=시간순, max_iter=50000))
월없는X = X선형.drop(columns=[c for c in X선형.columns if c.startswith("월_")])            # 달마다 따로 배우면 「몇 월인가」 열은 필요 없다

후보 = {"기온 × 시간대 선형회귀": ("선형회귀", 한번(선형, X선형.drop(columns=기온두열).join(시간대기온))),
        "릿지": ("선형회귀", 한번(릿지, X선형)),
        "라쏘": ("선형회귀", 한번(라쏘, X선형)),
        "굽은 선 선형회귀": ("선형회귀", 한번(선형, X선형.drop(columns=기온두열 + ["일사"]).join(굽은열))),
        "달마다 따로 — 선형회귀": ("선형회귀", 달마다따로(선형, 월없는X)),
        "달마다 따로 — 그래디언트 부스팅": ("그래디언트 부스팅", 달마다따로(부스팅, X트리)),
        "시간대별 기온을 더한 그래디언트 부스팅": ("그래디언트 부스팅", 한번(부스팅, X트리.join(시간대기온)))}

표1, 표2 = [], []
for 이름, (대상, 예) in 후보.items():
    후, 기 = R2(실제, 예), R2(실제, 기준예측[대상])                                      # 2024년 한 해로 모은 결정계수 — 후보 · 비교 대상
    표1.append({"후보": 이름, "비교 대상": 대상, "비교 대상 결정계수": f"{기:.4f}", "후보 결정계수": f"{후:.4f}",
                "차이": f"{후 - 기:+.4f}", "0.002 넘게 높나": "넘는다" if 후 - 기 > 0.002 else "못 넘는다"})
    나아진 = sum(R2(실제[칸], 예[칸]) > R2(실제[칸], 기준예측[대상][칸]) for _, 칸 in 달묶음)   # 달마다 결정계수가 비교 대상보다 높았던 달
    표2.append({"후보": 이름, "비교 대상": 대상, "11달 가운데 나아진 달": 나아진})
print("[표 1] 2023년으로 배워 2024년을 맞힌 결정계수 — 차이가 0.002 넘게 높으면 7단계에 올린다(6-가)")
display(pd.DataFrame(표1).set_index("후보"))
print("[표 2] 달마다 결정계수가 비교 대상보다 높았던 달 수 — 참고(8월이 빠진 11달)")
display(pd.DataFrame(표2).set_index("후보"))

[표 1] 2023년으로 배워 2024년을 맞힌 결정계수 — 차이가 0.002 넘게 높으면 7단계에 올린다(6-가)


,비교 대상,비교 대상 결정계수,후보 결정계수,차이,0.002 넘게 높나
후보,,,,,
기온 × 시간대 선형회귀,선형회귀,0.7524,0.7679,+0.0155,넘는다
릿지,선형회귀,0.7524,0.7524,-0.0001,못 넘는다
라쏘,선형회귀,0.7524,0.7522,-0.0002,못 넘는다
굽은 선 선형회귀,선형회귀,0.7524,0.7529,+0.0005,못 넘는다
달마다 따로 — 선형회귀,선형회귀,0.7524,0.7524,+0.0000,못 넘는다
달마다 따로 — 그래디언트 부스팅,그래디언트 부스팅,0.7807,0.7088,-0.0719,못 넘는다
시간대별 기온을 더한 그래디언트 부스팅,그래디언트 부스팅,0.7807,0.7800,-0.0007,못 넘는다


[표 2] 달마다 결정계수가 비교 대상보다 높았던 달 수 — 참고(8월이 빠진 11달)


,비교 대상,11달 가운데 나아진 달
후보,,
기온 × 시간대 선형회귀,선형회귀,10
릿지,선형회귀,6
라쏘,선형회귀,6
굽은 선 선형회귀,선형회귀,7
달마다 따로 — 선형회귀,선형회귀,7
달마다 따로 — 그래디언트 부스팅,그래디언트 부스팅,1
시간대별 기온을 더한 그래디언트 부스팅,그래디언트 부스팅,5


## 6-라 — 트리 계열의 설정을 2023년 안에서 고르면 2024년을 더 잘 맞히나

**그리드 서치는 트리 계열 두 모델(랜덤포레스트 · 그래디언트 부스팅)에만 씁니다.** 코드 23번째 줄의 `GridSearchCV`가 그리드 서치입니다 — `후보설정`에 적은 값들을 모두 조합해 하나씩 돌려 보고, 2023년 안에서 가장 잘 맞힌 조합을 고릅니다. 선형회귀는 고를 설정이 없어서, 자료를 넣으면 기울기가 하나로 계산되어 나옵니다(기온 × 시간대 선형회귀도 같습니다). 선형회귀 가운데 고를 설정이 있는 릿지·라쏘의 벌점 크기는 6-나에서 2023년 안에서 여러 값을 넣어 보고 골랐습니다 — 그리드 서치와 같은 방식입니다.



**무엇을 보는 셀인가** — [표 1]은 2023년 안에서 고른 설정, [표 2]는 지금 설정과 고른 설정으로 2024년을 맞힌 결정계수와 차이, [표 3]은 11달 가운데 몇 달 나아졌는지(참고)입니다.

In [368]:
# 6단계 분석계획 · 6-라
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit                  # 설정 후보를 모두 돌려 가장 나은 것을 고른다 · 시간 순서로 나누기
from sklearn.base import clone                                                      # 같은 모델 틀을 새로 하나 만든다

def R2(실, 예):                                                                      # 결정계수 — 5-다와 같은 식(준비 ①·② 다음에 이 셀만 돌려도 되게 다시 적는다)
    return 1 - ((실 - 예) ** 2).sum() / ((실 - 실.mean()) ** 2).sum()

실제 = y[맞힘]                                                                        # 2024년 실제 LNG(MWh)
달24 = 학습.loc[맞힘, "month"]                                                        # 2024년 시각마다 몇 월인지 — 8월은 없다
시간순 = TimeSeriesSplit(n_splits=4)                                                   # 2023년을 시간 순서로 넷으로 나눈다 — 설정은 2023년 안에서만 고른다
후보설정 = {"랜덤포레스트": (RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0),          # 나무 300그루 — 5-다와 같다
                          {"min_samples_leaf": [1, 5, 20], "max_features": [1.0, 0.5]},              # 바꿔 볼 값
                          {"min_samples_leaf": 5, "max_features": 1.0}),                             # 지금 설정(5-다)
            "그래디언트 부스팅": (HistGradientBoostingRegressor(early_stopping=False, random_state=0),
                             {"learning_rate": [0.05, 0.1], "max_leaf_nodes": [15, 31, 63],
                              "min_samples_leaf": [20, 50], "max_iter": [100, 300]},
                             {"learning_rate": 0.1, "max_leaf_nodes": 31, "min_samples_leaf": 20, "max_iter": 100})}
이름표 = {"min_samples_leaf": "잎 하나의 최소 시각 수", "max_features": "가지를 나눌 때마다 보는 열의 비율",
          "learning_rate": "배우는 속도", "max_leaf_nodes": "나무 하나의 잎 수", "max_iter": "나무 수"}

표1, 표2, 표3 = [], [], []
for 모델, (틀, 격자, 지금) in 후보설정.items():
    찾기 = GridSearchCV(틀, 격자, cv=시간순, scoring="r2").fit(X트리[배움], y[배움])       # 2023년 안에서만 고르고, 고른 설정으로 2023년 전체를 다시 배운다
    고른 = 찾기.best_params_
    표1 += [{"모델": 모델, "설정": 이름표[k], "지금": 지금[k], "고른 값": 고른[k]} for k in 격자]
    예지금 = pd.Series(clone(틀).set_params(**지금).fit(X트리[배움], y[배움]).predict(X트리[맞힘]), index=실제.index)  # 지금 설정으로 2024년을 맞힌다
    예고른 = pd.Series(찾기.best_estimator_.predict(X트리[맞힘]), index=실제.index)                                # 고른 설정으로 2024년을 맞힌다
    지, 고 = R2(실제, 예지금), R2(실제, 예고른)
    표2.append({"모델": 모델, "지금 설정": f"{지:.4f}", "고른 설정": f"{고:.4f}", "차이": f"{고 - 지:+.4f}",
                "0.002 넘게 높나": "넘는다" if 고 - 지 > 0.002 else "못 넘는다"})
    나아진 = sum(R2(실제[달24 == m], 예고른[달24 == m]) > R2(실제[달24 == m], 예지금[달24 == m]) for m in sorted(달24.unique()))
    표3.append({"모델": 모델, "11달 가운데 나아진 달": 나아진})

print("[표 1] 2023년 안에서 고른 설정 — 지금 설정과 나란히")
display(pd.DataFrame(표1).set_index(["모델", "설정"]))
print("[표 2] 2023년으로 배워 2024년을 맞힌 결정계수 — 차이가 0.002 넘게 높으면 7단계에서 고른 설정을 쓴다(6-다)")
display(pd.DataFrame(표2).set_index("모델"))
print("[표 3] 달마다 결정계수가 지금 설정보다 높았던 달 수 — 참고(8월이 빠진 11달)")
display(pd.DataFrame(표3).set_index("모델"))

KeyboardInterrupt: 

## 6-마 — 그리드 서치가 고른 설정은 왜 2024년에 더 낮게 나왔나

**무엇을 보는 셀인가** — 그리드 서치는 2023년 안에서 네 번 나눠 맞힌 결정계수의 평균(왼쪽 칸)이 가장 높은 설정을 고릅니다. 같은 두 설정(지금 설정 · 고른 설정)이 2024년에는 몇이었는지(오른쪽 칸)를 나란히 놓습니다.

In [ ]:
# 6단계 분석계획 · 6-마
from sklearn.model_selection import cross_val_score, TimeSeriesSplit               # 2023년 안에서 시간 순서로 나눠 맞힌 결정계수 · 나누기

def R2(실, 예):                                                                      # 결정계수 — 5-다와 같은 식(준비 ①·② 다음에 이 셀만 돌려도 되게 다시 적는다)
    return 1 - ((실 - 예) ** 2).sum() / ((실 - 실.mean()) ** 2).sum()

시간순 = TimeSeriesSplit(n_splits=4)                                                   # 6-라와 같은 나누기 — 2023년을 시간 순서로 넷
설정들 = {"랜덤포레스트": (lambda **k: RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0, **k),
                         {"지금 설정": {"min_samples_leaf": 5, "max_features": 1.0},                    # 5-다
                          "고른 설정": {"min_samples_leaf": 1, "max_features": 1.0}}),                  # 6-라 [표 1]에서 고른 값
          "그래디언트 부스팅": (lambda **k: HistGradientBoostingRegressor(early_stopping=False, random_state=0, **k),
                            {"지금 설정": {"learning_rate": 0.1, "max_leaf_nodes": 31, "min_samples_leaf": 20, "max_iter": 100},
                             "고른 설정": {"learning_rate": 0.05, "max_leaf_nodes": 63, "min_samples_leaf": 50, "max_iter": 300}})}
줄 = []
for 모델, (만들기, 둘) in 설정들.items():
    for 이름, 값 in 둘.items():
        안 = cross_val_score(만들기(**값), X트리[배움], y[배움], cv=시간순, scoring="r2").mean()     # 그리드 서치가 고를 때 본 숫자 — 2023년 안에서 네 번 맞힌 결정계수의 평균
        다음해 = R2(y[맞힘], 만들기(**값).fit(X트리[배움], y[배움]).predict(X트리[맞힘]))           # 2023년 전체로 배워 2024년을 맞힌 결정계수(6-라 [표 2]와 같은 숫자)
        줄.append({"모델": 모델, "설정": 이름, "2023년 안에서 맞힌 결정계수 (네 번 평균)": f"{안:.4f}", "2024년을 맞힌 결정계수": f"{다음해:.4f}"})
print("[표 1] 그리드 서치가 고를 때 본 숫자(왼쪽)와 2024년을 맞힌 숫자(오른쪽) — 같은 두 설정")
display(pd.DataFrame(줄).set_index(["모델", "설정"]))

[표 1] 그리드 서치가 고를 때 본 숫자(왼쪽)와 2024년을 맞힌 숫자(오른쪽) — 같은 두 설정


2023년 안에서 맞힌 결정계수 (네 번 평균) 2024년을 맞힌 결정계수
모델        설정                                             
랜덤포레스트    지금 설정                     0.5806         0.7618
          고른 설정                     0.5855         0.7632
그래디언트 부스팅 지금 설정                     0.6334         0.7807
          고른 설정                     0.6606         0.7790

## 6-바 — 트리 계열의 설정을 2023년으로 배워 2024년을 맞힌 결정계수로 고르면

**기준(6-다와 같음)** — 고른 설정이 지금 설정보다 2024년 결정계수가 0.002 넘게 높으면 7단계에서 그 설정을 씁니다. 고른 설정의 2024년 숫자는 여러 조합 가운데 가장 높은 것이라 조금 좋게 나온다는 것을 감안합니다. 2025년은 쓰지 않습니다.

**무엇을 보는 셀인가** — [표 1] 2024년으로 고른 설정, [표 2] 2024년 결정계수와 차이(해 본 조합 수 포함), [표 3] 11달 가운데 나아진 달(참고).

In [ ]:
# 6단계 분석계획 · 6-바
from itertools import product                                                       # 설정 값의 모든 조합을 만든다

def R2(실, 예):                                                                      # 결정계수 — 5-다와 같은 식(준비 ①·② 다음에 이 셀만 돌려도 되게 다시 적는다)
    return 1 - ((실 - 예) ** 2).sum() / ((실 - 실.mean()) ** 2).sum()

실제 = y[맞힘]                                                                        # 2024년 실제 LNG(MWh)
달24 = 학습.loc[맞힘, "month"]                                                        # 2024년 시각마다 몇 월인지 — 8월은 없다
격자들 = {"랜덤포레스트": (lambda **k: RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0, **k),   # 6-라와 같은 후보 값
                         {"min_samples_leaf": [1, 5, 20], "max_features": [1.0, 0.5]},
                         {"min_samples_leaf": 5, "max_features": 1.0}),                                        # 지금 설정(5-다)
          "그래디언트 부스팅": (lambda **k: HistGradientBoostingRegressor(early_stopping=False, random_state=0, **k),
                            {"learning_rate": [0.05, 0.1], "max_leaf_nodes": [15, 31, 63], "min_samples_leaf": [20, 50], "max_iter": [100, 300]},
                            {"learning_rate": 0.1, "max_leaf_nodes": 31, "min_samples_leaf": 20, "max_iter": 100})}
이름표 = {"min_samples_leaf": "잎 하나의 최소 시각 수", "max_features": "가지를 나눌 때마다 보는 열의 비율",
          "learning_rate": "배우는 속도", "max_leaf_nodes": "나무 하나의 잎 수", "max_iter": "나무 수"}

표1, 표2, 표3 = [], [], []
for 모델, (만들기, 격자, 지금) in 격자들.items():
    결과 = []                                                                         # (2024년 결정계수, 설정, 2024년 예측)
    for 값들 in product(*격자.values()):                                              # 조합마다 2023년으로 배워 2024년을 맞힌다 — 실제로 할 일(한 해로 배워 다음 해)과 같은 꼴
        설정 = dict(zip(격자, 값들))
        예 = pd.Series(만들기(**설정).fit(X트리[배움], y[배움]).predict(X트리[맞힘]), index=실제.index)
        결과.append((R2(실제, 예), 설정, 예))
    고, 고른, 예고른 = max(결과, key=lambda t: t[0])                                   # 2024년 결정계수가 가장 높은 조합
    지, _, 예지금 = [t for t in 결과 if t[1] == 지금][0]                                # 지금 설정
    표1 += [{"모델": 모델, "설정": 이름표[k], "지금": 지금[k], "2024년으로 고른 값": 고른[k]} for k in 격자]
    표2.append({"모델": 모델, "해 본 조합 수": len(결과), "지금 설정": f"{지:.4f}", "고른 설정": f"{고:.4f}",
                "차이": f"{고 - 지:+.4f}", "0.002 넘게 높나": "넘는다" if 고 - 지 > 0.002 else "못 넘는다"})
    나아진 = sum(R2(실제[달24 == m], 예고른[달24 == m]) > R2(실제[달24 == m], 예지금[달24 == m]) for m in sorted(달24.unique()))
    표3.append({"모델": 모델, "11달 가운데 나아진 달": 나아진})

print("[표 1] 2023년으로 배워 2024년을 가장 잘 맞힌 설정 — 지금 설정과 나란히")
display(pd.DataFrame(표1).set_index(["모델", "설정"]))
print("[표 2] 2024년 결정계수 — 고른 설정은 여러 조합 가운데 가장 높은 것이라 실제보다 조금 좋게 나온다(진짜 확인은 7단계 2025년)")
display(pd.DataFrame(표2).set_index("모델"))
print("[표 3] 달마다 결정계수가 지금 설정보다 높았던 달 수 — 참고(8월이 빠진 11달)")
display(pd.DataFrame(표3).set_index("모델"))

## 6-사 — 설정을 바꾸면 달마다 몇 MWh 덜 틀리나

**무엇을 보는 셀인가** — 2024년 달마다 평균 오차(한 시각에 평균 몇 MWh 틀렸나)를 지금 설정과 6-바에서 고른 설정으로 내고, 고른 설정이 얼마나 덜 틀렸는지 놓습니다. 맨 아래 줄은 2024년 한 해입니다.

In [ ]:
# 6단계 분석계획 · 6-사
실제 = y[맞힘]                                                                        # 2024년 실제 LNG(MWh)
달24 = 학습.loc[맞힘, "month"]                                                        # 2024년 시각마다 몇 월인지 — 8월은 없다
설정들 = {"랜덤포레스트": (lambda **k: RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0, **k),
                         {"min_samples_leaf": 5, "max_features": 1.0},                       # 지금 설정(5-다)
                         {"min_samples_leaf": 1, "max_features": 0.5}),                      # 6-바 [표 1]에서 2024년으로 고른 값
          "그래디언트 부스팅": (lambda **k: HistGradientBoostingRegressor(early_stopping=False, random_state=0, **k),
                            {"min_samples_leaf": 20},                                        # 지금 설정(5-다) — 나머지는 기본값
                            {"min_samples_leaf": 50})}                                       # 6-바 [표 1]에서 고른 값 — 바뀐 것은 이것 하나
열 = {}
for 모델, (만들기, 지금, 고른) in 설정들.items():
    오차지금 = (실제 - 만들기(**지금).fit(X트리[배움], y[배움]).predict(X트리[맞힘])).abs()     # 시각마다 지금 설정이 틀린 양(MWh)
    오차고른 = (실제 - 만들기(**고른).fit(X트리[배움], y[배움]).predict(X트리[맞힘])).abs()     # 시각마다 고른 설정이 틀린 양(MWh)
    달별 = pd.DataFrame({"지금": 오차지금, "고른": 오차고른}).groupby(달24.values).mean()      # 달마다 평균 오차
    달별.loc["2024년 한 해"] = [오차지금.mean(), 오차고른.mean()]                              # 한 해 전체 평균 오차
    열[(모델, "지금 설정의 평균 오차")] = 달별["지금"].round(0).astype(int)
    열[(모델, "고른 설정이 덜 틀린 양")] = (달별["지금"] - 달별["고른"]).round(0).astype(int)   # +면 고른 설정이 그만큼 덜 틀렸다
표 = pd.DataFrame(열)
표.index = [f"{i}월" if isinstance(i, (int, np.integer)) else i for i in 표.index]
print("[표 1] 달마다 평균 오차(MWh) — 「고른 설정이 덜 틀린 양」이 +면 고른 설정이 그만큼 덜 틀렸다")
display(표)

[표 1] 달마다 평균 오차(MWh) — 「고른 설정이 덜 틀린 양」이 +면 고른 설정이 그만큼 덜 틀렸다


랜덤포레스트                  그래디언트 부스팅              
          지금 설정의 평균 오차 고른 설정이 덜 틀린 양 지금 설정의 평균 오차 고른 설정이 덜 틀린 양
1월                1670           -22         1581           -26
2월                2308            11         2215            17
3월                1900           -21         1770            16
4월                1782             6         1752            12
5월                1807            73         1811            -8
6월                1999            42         1947             1
7월                2172             6         2092             6
9월                2633           161         2282            18
10월               2234            -3         2216           -13
11월               2254            73         2364            44
12월               2122           -18         2040           -60
2024년 한 해         2077            28         2004             0

## 6-아 — 6-바에서 고른 값보다 더 크거나 작은 값도 시험해 보기

**무엇을 보는 셀인가** — [표 1]은 6-아에서 더 시험한 값을 6-바에서 시험한 값과 나란히 놓은 것이고, [표 2]는 넓힌 후보 가운데 2024년 결정계수가 가장 높은 것과 물음 6에서 7단계에 올리기로 한 설정의 차이입니다. 0.002 넘게 높으면 바꾸고, 아니면 그대로 갑니다(6-다 기준). 그래디언트 부스팅의 나무 하나의 잎 수(31)는 6-바에서 가운데 값을 골랐으므로 그대로 둡니다. 준비 ①·② 다음에 이 셀만 돌리면 됩니다(1분 안쪽).

In [ ]:
# 6단계 분석계획 · 6-아
from itertools import product                                                       # 설정 값의 모든 조합을 만든다

def R2(실, 예):                                                                      # 결정계수 — 5-다와 같은 식(준비 ①·② 다음에 이 셀만 돌려도 되게 다시 적는다)
    return 1 - ((실 - 예) ** 2).sum() / ((실 - 실.mean()) ** 2).sum()

실제 = y[맞힘]                                                                        # 2024년 실제 LNG(MWh)
# 모델마다 (만들기, 넓힌 후보, 비교 대상 = 7단계에 올리기로 한 설정, 6-바에서 시험한 값)
넓힘 = {"랜덤포레스트": (lambda **k: RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0, **k),
                       {"min_samples_leaf": [1], "max_features": [3, 2, 1]},          # 6-바에서 고른 값(잎 1시각 · 7열 가운데 3열 = 비율 0.5)에서 열 수를 더 줄인다
                       {"min_samples_leaf": 1, "max_features": 3},                    # 비교 대상 — 6-바에서 고른 설정(물음 6 결론)
                       {"min_samples_leaf": "1 · 5 · 20", "max_features": "7열 · 3열(비율 1.0 · 0.5)"}),
        "그래디언트 부스팅": (lambda **k: HistGradientBoostingRegressor(early_stopping=False, random_state=0, max_leaf_nodes=31, **k),
                          {"learning_rate": [0.1, 0.2], "min_samples_leaf": [50, 100, 200], "max_iter": [100, 50]},   # 6-바에서 고른 값이 가장 크거나 작은 쪽이었던 셋을 그보다 더 크거나 작게
                          {"learning_rate": 0.1, "min_samples_leaf": 20, "max_iter": 100},                          # 비교 대상 — 지금 설정(물음 6 결론)
                          {"learning_rate": "0.05 · 0.1", "min_samples_leaf": "20 · 50", "max_iter": "100 · 300"})}
이름표 = {"min_samples_leaf": "잎 하나의 최소 시각 수", "max_features": "가지를 나눌 때마다 보는 열 수",
          "learning_rate": "배우는 속도", "max_iter": "나무 수"}

표1, 표2 = [], []
for 모델, (만들기, 후보, 비교, 전에) in 넓힘.items():
    표1 += [{"모델": 모델, "설정": 이름표[k], "6-바에서 시험한 값": 전에[k], "6-아에서 시험한 값": " · ".join(map(str, 후보[k]))} for k in 후보]
    예비교 = 만들기(**비교).fit(X트리[배움], y[배움]).predict(X트리[맞힘])            # 비교 대상을 2023년으로 배워 2024년을 맞힌다
    결과 = []                                                                         # (2024년 결정계수, 설정)
    for 값들 in product(*후보.values()):                                              # 넓힌 후보의 조합마다 같은 시험
        설정 = dict(zip(후보, 값들))
        결과.append((R2(실제, 만들기(**설정).fit(X트리[배움], y[배움]).predict(X트리[맞힘])), 설정))
    고, 고른 = max(결과, key=lambda t: t[0])                                           # 넓힌 후보 가운데 2024년 결정계수가 가장 높은 조합
    비 = R2(실제, 예비교)
    표2.append({"모델": 모델, "해 본 조합 수": len(결과), "비교 대상": f"{비:.4f}", "넓힌 후보 가운데 가장 높은 것": f"{고:.4f}",
                "차이": f"{고 - 비:+.4f}", "0.002 넘게 높나": "넘는다" if 고 - 비 > 0.002 else "못 넘는다",
                "그 설정": " · ".join(f"{이름표[k]} {v}" for k, v in 고른.items())})

print("[표 1] 더 시험한 값 — 6-바에서 고른 값이 시험한 값 가운데 가장 크거나 작은 값이었던 설정만, 그보다 더 크거나 작은 값을 더 시험했다")
display(pd.DataFrame(표1).set_index(["모델", "설정"]))
print("[표 2] 2023년으로 배워 2024년을 맞힌 결정계수 — 비교 대상은 물음 6에서 7단계에 올리기로 한 설정 · 후보가 많을수록 가장 높은 값은 조금 좋게 나온다(진짜 확인은 7단계 2025년)")
display(pd.DataFrame(표2).set_index("모델"))

[표 1] 더 시험한 값 — 6-바에서 고른 값이 시험한 값 가운데 가장 크거나 작은 값이었던 설정만, 그보다 더 크거나 작은 값을 더 시험했다


6-바에서 시험한 값           6-아에서 시험한 값
모델        설정                                                      
랜덤포레스트    잎 하나의 최소 시각 수                 1 · 5 · 20               1
          가지를 나눌 때마다 보는 열 수  7열 · 3열(비율 1.0 · 0.5)       3 · 2 · 1
그래디언트 부스팅 배우는 속도                        0.05 · 0.1       0.1 · 0.2
          잎 하나의 최소 시각 수                    20 · 50  50 · 100 · 200
          나무 수                           100 · 300        100 · 50

[표 2] 2023년으로 배워 2024년을 맞힌 결정계수 — 비교 대상은 물음 6에서 7단계에 올리기로 한 설정 · 후보가 많을수록 가장 높은 값은 조금 좋게 나온다(진짜 확인은 7단계 2025년)


,해 본 조합 수,비교 대상,넓힌 후보 가운데 가장 높은 것,차이,0.002 넘게 높나,그 설정
모델,,,,,,
랜덤포레스트,3,0.7701,0.7701,+0.0000,못 넘는다,잎 하나의 최소 시각 수 1 · 가지를 나눌 때마다 보는 열 수 3
그래디언트 부스팅,12,0.7807,0.7824,+0.0017,못 넘는다,배우는 속도 0.1 · 잎 하나의 최소 시각 수 50 · 나무 수 100


# 물음 7 — 시스템 다이내믹스 인과지도: 무엇이 LNG를 움직이고, 모델은 그 가운데 무엇을 보나

> **결정** — 모델은 LNG를 바로 움직이는 여섯을 하나도 보지 않고, 날씨·시간·달·날 종류만 본다